In [3]:
"""
sklearn_root_only_4_designs_SAME_BB_SPLIT_HYPERPARAMS.py

ROOT-NODE-ONLY sklearn experiment.

Exactly FOUR experimental designs:
    1) Combined  -> Combined   Classification
    2) Combined  -> Combined   Regression
    3) Synthetic'-> Combined   Classification
    4) Synthetic'-> Combined   Regression

Each design fits the same 3 sklearn models used in the final B&B experiment.
Therefore:
    4 experimental designs
    12 fitted model results

NO branch-and-bound information is used as model input.
NO dynamic features.
NO node-count filtering.
Input = exactly the same 64 static DI features used in the B&B static runs.

Consistency with the final B&B sklearn experiment
==================================================
Locked to the B&B experiment:
- same Combined eligible set (280)
- same Synthetic' eligible set (6069)
- same 3 explicit Combined exclusions
- same exact Combined train/test split manifest for Combined->Combined
- same Synthetic'->Combined semantics as the B&B runner:
      train = all 6069 Synthetic'
      test  = all 280 Combined
- same 64 static features
- same median imputation
- no scaling
- same random seed 42
- same XGB / GradientBoosting / RandomForest hyperparameters
- same classification/regression metrics

ONLY the target changes.

Root regression target:
    delta_root =
        (lp_root_obj - sdp_root_obj)
        / (abs(lp_root_obj) + abs(sdp_root_obj) + 1e-3)

Root classification target:
    y_root = 1 if delta_root > -0.01 else 0
    1 = LP
    0 = SDP_PRIME

Root objectives are read from the CLEAN B&B summary files only as a convenient
stored source of each instance's root LP and SDP' relaxation values. No tree,
node-budget, frontier, depth, or dynamic quantity enters the model.

The script hard-fails if the root summaries do not cover exactly the SAME
eligible B&B instances. It never silently drops instances.
"""

from __future__ import annotations

from datetime import datetime
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Tuple

import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import (
    GradientBoostingClassifier,
    GradientBoostingRegressor,
    RandomForestClassifier,
    RandomForestRegressor,
)
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

try:
    from xgboost import XGBClassifier, XGBRegressor
except Exception as e:
    raise ImportError(
        "xgboost is required because the final B&B experiment used XGB-C/XGB-R."
    ) from e


# ============================================================
# CONFIG — LOCKED TO FINAL B&B EXPERIMENT
# ============================================================

ROOT = Path(r"C:\Users\suuser\Desktop\minlplib")
FINAL_DIR = ROOT / "Areview" / "combined_minlplib_qplib"
DIM_TAG = "combined_minlplib_qplib"
SYNTHETIC_FINAL_DIR = (
    ROOT / "paperexperiments" / "synthetic_prime_minimal_revision"
    / "bb_separate_trees" / "synthetic_prime_8016_fixed100_v1"
    / "postrun_clean"
)

SKLEARN_RUN_NAME = "sklearn_all_runs_synthetic_prime_8016_20261021_014902_034048"


RANDOM_STATE = 42
TARGET_COL = "delta_root"
CLASS_COL = "y_root"

TAU = 0.01
ETA = 1e-3

USE_SAMPLE_WEIGHTS = False
WEIGHT_ALPHA = 0.30

EXPECTED_STATIC_FEATURES = 64
EXPECTED_COMBINED = 280
EXPECTED_SYNTHETIC = 6069

EXCLUDE_COMBINED_KEYS = {
    "QPLIB||QPLIB_2512",
    "QPLIB||QPLIB_3402",
    "QPLIB||QPLIB_3703",
}

# Use the same final B&B target files ONLY to recover the exact eligible
# instance sets. Their B&B delta/y values are NOT used as root targets.
BB_TARGET_PROBE = 10

COMBINED_BB_TARGET = (
    FINAL_DIR
    / "targets"
    / "method2"
    / DIM_TAG
    / f"targets_method2_relaxation_{DIM_TAG}_probe{BB_TARGET_PROBE}.csv"
)

SYNTHETIC_BB_TARGET = (SYNTHETIC_FINAL_DIR / "targets" / "method2"
                       / "synthetic_prime" / f"targets_method2_relaxation_synthetic_prime_probe{BB_TARGET_PROBE}.csv")

# Same static feature files as final B&B static runs.
COMBINED_STATIC = (
    FINAL_DIR
    / "combined_minlplib_qplib_features_DI_64_clean.xlsx"
)

SYNTHETIC_STATIC = SYNTHETIC_FINAL_DIR / "synthetic_prime_DI_64_clean.xlsx"

# Root objectives stored in the clean summary files.
COMBINED_SUMMARY = (
    FINAL_DIR
    / "summary_combined_minlplib_qplib_BB_clean.csv"
)

SYNTHETIC_SUMMARY = SYNTHETIC_FINAL_DIR / "synthetic_prime_BB_summary.csv"

# SAME exact Combined split generated by the final B&B sklearn runner.
BB_OUTPUT_ROOT = FINAL_DIR / SKLEARN_RUN_NAME

COMBINED_SPLIT_FILE = (
    BB_OUTPUT_ROOT
    / "combined_train_test_split_manifest.csv"
)

RUN_STAMP = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
OUTPUT_ROOT = FINAL_DIR / f"sklearn_root_synprime_8016_{RUN_STAMP}"

MASTER_RESULTS_FILE = (
    OUTPUT_ROOT
    / "all_root_only_model_results.csv"
)

DESIGN_RESULTS_FILE = (
    OUTPUT_ROOT
    / "root_only_design_summary.csv"
)

EXPERIMENT_MANIFEST_FILE = (
    OUTPUT_ROOT
    / "experiment_manifest.csv"
)


# ============================================================
# HELPERS
# ============================================================

def _ts():
    return datetime.now().strftime("%H:%M:%S")


def read_any(path: Path) -> pd.DataFrame:
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f"Required file not found:\n{path}")

    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)

    if path.suffix.lower() in {".xlsx", ".xls"}:
        return pd.read_excel(path)

    raise ValueError(f"Unsupported file type: {path}")


def _canon_name(x) -> str:
    s = str(x).strip()
    if s.lower().endswith(".pkl"):
        s = s[:-4]
    return s


def _norm_source(x) -> str:
    return str(x).strip().upper()


def _require_columns(df: pd.DataFrame, cols: Iterable[str], label: str):
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ValueError(
            f"{label} missing columns {missing}. "
            f"Available columns: {list(df.columns)}"
        )


def _assert_unique(df: pd.DataFrame, key: str, label: str):
    if df[key].duplicated().any():
        ex = (
            df.loc[df[key].duplicated(keep=False), key]
            .astype(str)
            .head(20)
            .tolist()
        )
        raise ValueError(f"{label}: duplicate {key}. Examples: {ex}")


# ============================================================
# KEY HANDLING — SAME CONVENTIONS AS FINAL B&B SCRIPT
# ============================================================

def canonicalize_combined_key(df: pd.DataFrame, label: str) -> pd.DataFrame:
    out = df.copy()

    if "instance_key" in out.columns:
        out["_key"] = out["instance_key"].astype(str).str.strip()
    else:
        if "instance_name" not in out.columns:
            raise ValueError(
                f"{label}: need instance_key or instance_name."
            )

        out["instance_name"] = out["instance_name"].map(_canon_name)

        if "source" in out.columns:
            src = out["source"].map(_norm_source)

        elif "id" in out.columns:
            ids = out["id"].astype(str).str.strip()

            src = pd.Series(
                np.where(
                    ids.str.startswith("MINLPLIB__"),
                    "MINLPLIB",
                    np.where(
                        ids.str.startswith("QPLIB__"),
                        "QPLIB",
                        "",
                    ),
                ),
                index=out.index,
                dtype=object,
            )

            if (src == "").any():
                bad = out.loc[src == "", "id"].head(20).tolist()
                raise ValueError(
                    f"{label}: cannot infer source from id. Examples: {bad}"
                )
        else:
            raise ValueError(f"{label}: cannot construct Combined key.")

        out["_key"] = (
            src.astype(str)
            + "||"
            + out["instance_name"].astype(str)
        )

    out["_key"] = out["_key"].astype(str).str.strip()

    if "source" not in out.columns:
        out["source"] = out["_key"].str.split("||", n=1).str[0]

    out["source"] = out["source"].map(_norm_source)

    if "instance_name" not in out.columns:
        out["instance_name"] = out["_key"].str.split("||", n=1).str[1]

    out["instance_name"] = out["instance_name"].map(_canon_name)

    _assert_unique(out, "_key", label)
    return out


def canonicalize_synthetic_key(
    df: pd.DataFrame, label: str, target_keys: Optional[set] = None
) -> pd.DataFrame:
    """Canonical SYNTHETIC||<original-id>; never join by row positions."""
    out = df.copy()
    candidates = [c for c in ("instance_key", "dataset_id", "instance_id", "id")
                  if c in out.columns]
    if not candidates:
        raise ValueError(f"{label}: no original numeric ID or instance_key")
    best_col, best_keys, best_overlap = None, None, -1
    for col in candidates:
        raw = out[col].astype("string").str.strip()
        raw = raw.str.replace(r"^SYNTHETIC\|\|", "", regex=True)
        if raw.isna().any() or not raw.str.fullmatch(r"[0-9]+").all():
            continue
        keys = "SYNTHETIC||" + raw.astype(str)
        overlap = len(set(keys) & target_keys) if target_keys is not None else len(keys)
        if overlap > best_overlap:
            best_col, best_keys, best_overlap = col, keys, overlap
    if best_keys is None or (target_keys is not None and best_overlap == 0):
        raise ValueError(f"{label}: cannot align original Synthetic' IDs to targets")
    out["_key"] = best_keys
    _assert_unique(out, "_key", label)
    print(f"[{_ts()}] [{label}] matching on {best_col}; overlap={best_overlap}")
    return out


# ============================================================
# EXACT B&B ELIGIBLE INSTANCE SETS
# ============================================================

def load_bb_eligible_keys() -> Tuple[pd.DataFrame, pd.DataFrame]:
    combined = canonicalize_combined_key(
        read_any(COMBINED_BB_TARGET),
        "Combined final B&B target",
    )

    synthetic = canonicalize_synthetic_key(
        read_any(SYNTHETIC_BB_TARGET),
        "Synthetic' final B&B target",
    )

    # Same explicit three Combined exclusions as the final B&B experiment.
    combined = (
        combined.loc[
            ~combined["_key"].isin(EXCLUDE_COMBINED_KEYS)
        ]
        .copy()
        .reset_index(drop=True)
    )

    if len(combined) != EXPECTED_COMBINED:
        raise RuntimeError(
            f"Expected exactly {EXPECTED_COMBINED} final Combined B&B instances, "
            f"found {len(combined)}."
        )

    if len(synthetic) != EXPECTED_SYNTHETIC:
        raise RuntimeError(
            f"Expected exactly {EXPECTED_SYNTHETIC} final Synthetic' B&B instances, "
            f"found {len(synthetic)}."
        )

    _assert_unique(combined, "_key", "Combined B&B eligible set")
    _assert_unique(synthetic, "_key", "Synthetic' B&B eligible set")

    print(
        f"[{_ts()}] Locked B&B eligible sets: "
        f"Combined={len(combined)}, Synthetic'={len(synthetic)}",
        flush=True,
    )

    return combined, synthetic


# ============================================================
# ROOT TARGETS
# ============================================================

def compute_root_targets(
    summary_df: pd.DataFrame,
    dataset_kind: str,
    eligible_keys: set,
) -> pd.DataFrame:
    if dataset_kind == "combined":
        s = canonicalize_combined_key(
            summary_df,
            "Combined root summary",
        )
    elif dataset_kind == "synthetic":
        s = canonicalize_synthetic_key(
            summary_df,
            "Synthetic' root summary",
            target_keys=eligible_keys,
        )
    else:
        raise ValueError(dataset_kind)

    _require_columns(
        s,
        ["_key", "lp_root_obj", "sdp_root_obj"],
        f"{dataset_kind} root summary",
    )

    # Restrict to exactly the same final B&B population.
    missing = eligible_keys - set(s["_key"])
    if missing:
        raise RuntimeError(
            f"{dataset_kind}: root summary does not cover the exact B&B set. "
            f"Missing examples: {sorted(missing)[:30]}"
        )

    out = (
        s.loc[s["_key"].isin(eligible_keys)]
        .copy()
        .reset_index(drop=True)
    )

    if len(out) != len(eligible_keys):
        raise RuntimeError(
            f"{dataset_kind}: expected {len(eligible_keys)} root rows, "
            f"found {len(out)}."
        )

    out["lp_root_obj"] = pd.to_numeric(
        out["lp_root_obj"],
        errors="coerce",
    )

    out["sdp_root_obj"] = pd.to_numeric(
        out["sdp_root_obj"],
        errors="coerce",
    )

    finite = (
        np.isfinite(out["lp_root_obj"].to_numpy(float))
        & np.isfinite(out["sdp_root_obj"].to_numpy(float))
    )

    if not finite.all():
        bad = out.loc[
            ~finite,
            ["_key", "lp_root_obj", "sdp_root_obj"],
        ].head(30)

        raise RuntimeError(
            f"{dataset_kind}: some exact B&B instances do not have finite "
            "LP/SDP' root objectives. We do NOT silently drop them because "
            "the root experiment must use the same population.\n"
            + bad.to_string(index=False)
        )

    zlp = out["lp_root_obj"].to_numpy(float)
    zsdp = out["sdp_root_obj"].to_numpy(float)

    out[TARGET_COL] = (
        (zlp - zsdp)
        / (
            np.abs(zlp)
            + np.abs(zsdp)
            + ETA
        )
    )

    out[CLASS_COL] = (
        out[TARGET_COL].to_numpy(float)
        > -TAU
    ).astype(int)

    out["root_winner_thresholded"] = np.where(
        out[CLASS_COL].to_numpy(int) == 1,
        "LP",
        "SDP_PRIME",
    )

    keep_cols = [
        c
        for c in [
            "_key",
            "source",
            "instance_index",
            "instance_id",
            "instance_name",
            "lp_root_obj",
            "sdp_root_obj",
            TARGET_COL,
            CLASS_COL,
            "root_winner_thresholded",
        ]
        if c in out.columns
    ]

    out = out[keep_cols].copy()

    _assert_unique(out, "_key", f"{dataset_kind} root targets")

    print(
        f"[{_ts()}] [{dataset_kind}] root labels: "
        f"{out[CLASS_COL].value_counts().sort_index().to_dict()}",
        flush=True,
    )

    return out


# ============================================================
# STATIC FEATURE LOADING — EXACT SAME 64 FEATURES
# ============================================================

FORBIDDEN_STATIC_COLS = {
    "id",
    "id_key",
    "id_global",
    "instance_id",
    "instanceid",
    "instance_index",
    "instance_name",
    "instance_key",
    "problem_name",
    "name",
    "source",
    "_key",
    "y",
    "winner",
    "delta",
    "y_root",
    "delta_root",
    "lp_obj",
    "sdp_prime_obj",
    "sdp_obj",
    "sdpp_obj",
    "lp_root_obj",
    "sdp_root_obj",
    "lp_status",
    "sdp_status",
    "sdp_prime_status",
    "sdpp_status",
}


def load_static_features(
    dataset_kind: str,
    eligible_keys: set,
) -> Tuple[pd.DataFrame, List[str]]:
    path = (
        COMBINED_STATIC
        if dataset_kind == "combined"
        else SYNTHETIC_STATIC
    )

    raw = read_any(path)

    if dataset_kind == "combined":
        df = canonicalize_combined_key(
            raw,
            "Combined static features",
        )
    else:
        df = canonicalize_synthetic_key(
            raw,
            "Synthetic' static features",
            target_keys=eligible_keys,
        )

    feature_cols = [
        c
        for c in df.columns
        if c not in FORBIDDEN_STATIC_COLS
    ]

    if len(feature_cols) != EXPECTED_STATIC_FEATURES:
        raise RuntimeError(
            f"{dataset_kind}: expected exactly {EXPECTED_STATIC_FEATURES} "
            f"static DI features, found {len(feature_cols)}.\n"
            f"Columns: {feature_cols}"
        )

    missing = eligible_keys - set(df["_key"])
    if missing:
        raise RuntimeError(
            f"{dataset_kind}: static feature file misses B&B-eligible instances. "
            f"Examples: {sorted(missing)[:30]}"
        )

    out = (
        df.loc[df["_key"].isin(eligible_keys), ["_key"] + feature_cols]
        .copy()
        .reset_index(drop=True)
    )

    for c in feature_cols:
        out[c] = pd.to_numeric(out[c], errors="coerce")

    _assert_unique(out, "_key", f"{dataset_kind} static features")

    return out, feature_cols


# ============================================================
# SAME B&B COMBINED SPLIT
# ============================================================

def load_same_combined_split(
    combined_keys: set,
) -> Tuple[set, set, pd.DataFrame]:
    m = read_any(COMBINED_SPLIT_FILE)

    _require_columns(
        m,
        ["_key", "split"],
        "final B&B Combined split manifest",
    )

    m["_key"] = m["_key"].astype(str).str.strip()
    m["split"] = m["split"].astype(str).str.strip().str.lower()

    _assert_unique(m, "_key", "final B&B Combined split manifest")

    if set(m["_key"]) != combined_keys:
        raise RuntimeError(
            "The B&B split manifest does not match the final 280-instance "
            "Combined set."
        )

    if set(m["split"]) - {"train", "test"}:
        raise RuntimeError(
            "B&B split manifest contains values other than train/test."
        )

    train_keys = set(m.loc[m["split"] == "train", "_key"])
    test_keys = set(m.loc[m["split"] == "test", "_key"])

    if train_keys & test_keys:
        raise RuntimeError("B&B split manifest has train/test overlap.")

    print(
        f"[{_ts()}] Reusing exact B&B Combined split: "
        f"train={len(train_keys)}, test={len(test_keys)}",
        flush=True,
    )

    return train_keys, test_keys, m


# ============================================================
# MODELS — EXACT SAME HYPERPARAMETERS AS FINAL B&B RUNNER
# ============================================================

def make_models(task: str) -> Dict[str, object]:
    if task == "classification":
        return {
            "XGB-C": XGBClassifier(
                n_estimators=400,
                learning_rate=0.08,
                max_depth=6,
                subsample=0.8,
                colsample_bytree=0.8,
                reg_alpha=0.1,
                reg_lambda=1.0,
                random_state=RANDOM_STATE,
                n_jobs=-1,
                eval_metric="logloss",
            ),
            "GBC": GradientBoostingClassifier(
                random_state=RANDOM_STATE,
            ),
            "RFC": RandomForestClassifier(
                n_estimators=400,
                max_features="sqrt",
                class_weight="balanced",
                n_jobs=-1,
                random_state=RANDOM_STATE,
            ),
        }

    if task == "regression":
        return {
            "XGB-R": XGBRegressor(
                n_estimators=400,
                learning_rate=0.08,
                max_depth=6,
                subsample=0.8,
                colsample_bytree=0.8,
                reg_alpha=0.1,
                reg_lambda=1.0,
                random_state=RANDOM_STATE,
                n_jobs=-1,
            ),
            "GBR": GradientBoostingRegressor(
                random_state=RANDOM_STATE,
            ),
            "RFR": RandomForestRegressor(
                n_estimators=400,
                n_jobs=-1,
                random_state=RANDOM_STATE,
            ),
        }

    raise ValueError(f"Unknown task={task}")


def build_pipeline(model) -> Pipeline:
    try:
        imputer = SimpleImputer(
            strategy="median",
            keep_empty_features=True,
        )
    except TypeError:
        imputer = SimpleImputer(
            strategy="median",
        )

    return Pipeline(
        [
            ("imputer", imputer),
            ("model", model),
        ]
    )


# ============================================================
# METRICS — SAME AS FINAL B&B RUNNER
# ============================================================

def safe_pearson_r(y_true, y_pred):
    yt = np.asarray(y_true, dtype=float)
    yp = np.asarray(y_pred, dtype=float)

    mask = np.isfinite(yt) & np.isfinite(yp)
    yt = yt[mask]
    yp = yp[mask]

    if len(yt) < 2 or np.std(yt) <= 0 or np.std(yp) <= 0:
        return float("nan")

    r = np.corrcoef(yt, yp)[0, 1]
    return float(r) if np.isfinite(r) else float("nan")


def r_accuracy_paper(y_true_delta, y_pred_delta, eps=TAU):
    yt = np.asarray(y_true_delta, dtype=float)
    yp = np.asarray(y_pred_delta, dtype=float)

    mask = np.isfinite(yt) & np.isfinite(yp)
    yt = yt[mask]
    yp = yp[mask]

    if len(yt) == 0:
        return float("nan")

    same_sign = np.sign(yt) == np.sign(yp)

    forgiveness = (
        (yt > -eps)
        & (yt < 0)
        & (yp > 0)
    )

    return float(np.mean(same_sign | forgiveness))


def r_accuracy_tol(y_true_delta, y_pred_delta, eps=TAU):
    yt = np.asarray(y_true_delta, dtype=float)
    yp = np.asarray(y_pred_delta, dtype=float)

    mask = np.isfinite(yt) & np.isfinite(yp)
    yt = yt[mask]
    yp = yp[mask]

    if len(yt) == 0:
        return float("nan")

    same = np.sign(yt) == np.sign(yp)

    q2 = (
        (yt > -eps)
        & (yt < 0)
        & (yp > 0)
    )

    q4 = (
        (yt > 0)
        & (yt < eps)
        & (yp < 0)
        & (yp > -eps)
    )

    return float(np.mean(same | q2 | q4))


def decision_from_delta(delta_values):
    return (
        np.asarray(delta_values, dtype=float)
        > -TAU
    ).astype(int)


# ============================================================
# MERGE
# ============================================================

def merge_targets_features(
    target_df: pd.DataFrame,
    feature_df: pd.DataFrame,
    feature_cols: List[str],
    label: str,
) -> pd.DataFrame:
    missing = set(target_df["_key"]) - set(feature_df["_key"])

    if missing:
        raise RuntimeError(
            f"{label}: targets missing static features. "
            f"Examples: {sorted(missing)[:30]}"
        )

    target_cols = [
        c
        for c in [
            "_key",
            "source",
            "instance_index",
            "instance_id",
            "instance_name",
            "lp_root_obj",
            "sdp_root_obj",
            TARGET_COL,
            CLASS_COL,
        ]
        if c in target_df.columns
    ]

    out = target_df[target_cols].merge(
        feature_df[["_key"] + feature_cols],
        on="_key",
        how="left",
        validate="one_to_one",
    )

    if len(out) != len(target_df):
        raise RuntimeError(f"{label}: merge changed row count.")

    return out


# ============================================================
# ONE DESIGN
# ============================================================

def run_design(
    *,
    design_id: int,
    train_setting: str,
    task: str,
    combined_root: pd.DataFrame,
    synthetic_root: pd.DataFrame,
    combined_features: pd.DataFrame,
    synthetic_features: pd.DataFrame,
    feature_cols: List[str],
    combined_train_keys: set,
    combined_test_keys: set,
) -> List[dict]:
    if train_setting == "combined_to_combined":
        all_combined = merge_targets_features(
            combined_root,
            combined_features,
            feature_cols,
            "Combined root",
        )

        train_df = (
            all_combined.loc[
                all_combined["_key"].isin(combined_train_keys)
            ]
            .copy()
            .reset_index(drop=True)
        )

        test_df = (
            all_combined.loc[
                all_combined["_key"].isin(combined_test_keys)
            ]
            .copy()
            .reset_index(drop=True)
        )

        setting_label = "Combined→Combined"

    elif train_setting == "synthetic_to_combined":
        # IMPORTANT: exactly mirrors the final B&B sklearn runner.
        # Train on ALL Synthetic'; test on ALL 280 Combined.
        train_df = merge_targets_features(
            synthetic_root,
            synthetic_features,
            feature_cols,
            "Synthetic' root train",
        )

        test_df = merge_targets_features(
            combined_root,
            combined_features,
            feature_cols,
            "Combined root transfer test",
        )

        setting_label = "Synthetic′→Combined"

    else:
        raise ValueError(train_setting)

    if task == "classification":
        y_train = train_df[CLASS_COL].to_numpy(int)
        y_test = test_df[CLASS_COL].to_numpy(int)

        if len(np.unique(y_train)) < 2:
            raise RuntimeError(
                f"Design {design_id}: classification train set is single-class."
            )
    else:
        y_train = None
        y_test = test_df[CLASS_COL].to_numpy(int)

    d_train = train_df[TARGET_COL].to_numpy(float)
    d_test = test_df[TARGET_COL].to_numpy(float)

    X_train = train_df[feature_cols].replace([np.inf, -np.inf], np.nan)
    X_test = test_df[feature_cols].replace([np.inf, -np.inf], np.nan)

    run_dir = (
        OUTPUT_ROOT
        / f"design_{design_id:02d}__{train_setting}__{task}"
    )

    run_dir.mkdir(parents=True, exist_ok=True)

    split_out = pd.concat(
        [
            train_df[["_key", TARGET_COL, CLASS_COL]].assign(split="train"),
            test_df[["_key", TARGET_COL, CLASS_COL]].assign(split="test"),
        ],
        ignore_index=True,
    )

    split_out.to_csv(
        run_dir / "instance_split_root_targets.csv",
        index=False,
    )

    pd.DataFrame({"feature": feature_cols}).to_csv(
        run_dir / "feature_columns.csv",
        index=False,
    )

    print("\n" + "=" * 88)
    print(
        f"[{_ts()}] DESIGN {design_id} | {setting_label} | ROOT {task.upper()}"
    )
    print(
        f"[{_ts()}] train={len(train_df)} test={len(test_df)} "
        f"features={len(feature_cols)}"
    )
    print(
        f"[{_ts()}] root train labels="
        f"{train_df[CLASS_COL].value_counts().sort_index().to_dict()} | "
        f"test labels="
        f"{test_df[CLASS_COL].value_counts().sort_index().to_dict()}"
    )
    print("=" * 88)

    result_rows = []

    for model_name, model in make_models(task).items():
        print(f"[{_ts()}] training {model_name}...", flush=True)

        pipe = build_pipeline(model)

        if task == "classification":
            pipe.fit(X_train, y_train)

            pred = pipe.predict(X_test).astype(int)

            if hasattr(pipe, "predict_proba") and len(np.unique(y_test)) > 1:
                prob = pipe.predict_proba(X_test)[:, 1]
                auc = float(roc_auc_score(y_test, prob))
            else:
                prob = np.full(len(y_test), np.nan)
                auc = float("nan")

            acc = float(accuracy_score(y_test, pred))
            bal = float(balanced_accuracy_score(y_test, pred))
            f1_lp = float(f1_score(y_test, pred, zero_division=0))
            f1_macro = float(
                f1_score(
                    y_test,
                    pred,
                    average="macro",
                    zero_division=0,
                )
            )
            precision = float(
                precision_score(y_test, pred, zero_division=0)
            )
            recall = float(
                recall_score(y_test, pred, zero_division=0)
            )

            cm = confusion_matrix(y_test, pred, labels=[0, 1])

            pd.DataFrame(
                cm,
                index=["true_SDP_PRIME(0)", "true_LP(1)"],
                columns=["pred_SDP_PRIME(0)", "pred_LP(1)"],
            ).to_csv(
                run_dir / f"cm_{model_name}.csv",
                index=True,
            )

            disp = ConfusionMatrixDisplay(
                cm,
                display_labels=["SDP_PRIME", "LP"],
            )
            disp.plot()
            plt.title(
                f"Root | {setting_label} | {model_name}"
            )
            plt.tight_layout()
            plt.savefig(
                run_dir / f"cm_{model_name}.png",
                dpi=200,
            )
            plt.close()

            pred_df = test_df[
                ["_key", TARGET_COL, CLASS_COL]
            ].copy()

            pred_df["y_pred"] = pred
            pred_df["prob_LP"] = prob

            pred_df.to_csv(
                run_dir / f"preds_{model_name}.csv",
                index=False,
            )

            result = {
                "design_id": design_id,
                "train_test": setting_label,
                "task": task,
                "input": "64_static_DI",
                "target": "root_LP_vs_SDP_PRIME",
                "model": model_name,
                "n_train": len(train_df),
                "n_test": len(test_df),
                "accuracy": acc,
                "balanced_accuracy": bal,
                "f1_lp": f1_lp,
                "f1_macro": f1_macro,
                "precision_LP": precision,
                "recall_LP": recall,
                "roc_auc": auc,
                "rmse": np.nan,
                "mae": np.nan,
                "r2": np.nan,
                "pearson_r": np.nan,
                "r_accuracy_paper": np.nan,
                "r_accuracy_tol": np.nan,
                "decision_accuracy": np.nan,
            }

            print(
                f"[{_ts()}] {model_name}: "
                f"acc={acc:.4f} bal={bal:.4f} macroF1={f1_macro:.4f}",
                flush=True,
            )

        else:
            pipe.fit(X_train, d_train)

            delta_pred = pipe.predict(X_test)
            pred_class = decision_from_delta(delta_pred)

            mse = float(
                mean_squared_error(
                    d_test,
                    delta_pred,
                )
            )
            rmse = float(np.sqrt(mse))
            mae = float(
                mean_absolute_error(
                    d_test,
                    delta_pred,
                )
            )
            r2 = float(
                r2_score(
                    d_test,
                    delta_pred,
                )
            )
            pearson = safe_pearson_r(
                d_test,
                delta_pred,
            )
            r_paper = r_accuracy_paper(
                d_test,
                delta_pred,
            )
            r_tol = r_accuracy_tol(
                d_test,
                delta_pred,
            )
            decision_acc = float(
                accuracy_score(
                    y_test,
                    pred_class,
                )
            )

            pred_df = test_df[
                ["_key", TARGET_COL, CLASS_COL]
            ].copy()

            pred_df["delta_pred"] = delta_pred
            pred_df["y_pred_from_delta"] = pred_class

            pred_df.to_csv(
                run_dir / f"preds_{model_name}.csv",
                index=False,
            )

            result = {
                "design_id": design_id,
                "train_test": setting_label,
                "task": task,
                "input": "64_static_DI",
                "target": "root_LP_vs_SDP_PRIME",
                "model": model_name,
                "n_train": len(train_df),
                "n_test": len(test_df),
                "accuracy": np.nan,
                "balanced_accuracy": np.nan,
                "f1_lp": np.nan,
                "f1_macro": np.nan,
                "precision_LP": np.nan,
                "recall_LP": np.nan,
                "roc_auc": np.nan,
                "rmse": rmse,
                "mae": mae,
                "r2": r2,
                "pearson_r": pearson,
                "r_accuracy_paper": r_paper,
                "r_accuracy_tol": r_tol,
                "decision_accuracy": decision_acc,
            }

            print(
                f"[{_ts()}] {model_name}: "
                f"rAcc_paper={r_paper:.4f} "
                f"rAcc_tol={r_tol:.4f} "
                f"RMSE={rmse:.5f} MAE={mae:.5f} R2={r2:.4f}",
                flush=True,
            )

        joblib.dump(
            pipe,
            run_dir / f"model_{model_name}.joblib",
        )

        result_rows.append(result)

    pd.DataFrame(result_rows).to_csv(
        run_dir / "results_all_models.csv",
        index=False,
    )

    return result_rows


# ============================================================
# MAIN
# ============================================================

def main():
    required = [
        COMBINED_BB_TARGET,
        SYNTHETIC_BB_TARGET,
        COMBINED_STATIC,
        SYNTHETIC_STATIC,
        COMBINED_SUMMARY,
        SYNTHETIC_SUMMARY,
        COMBINED_SPLIT_FILE,
    ]

    missing = [str(p) for p in required if not Path(p).exists()]

    if missing:
        raise FileNotFoundError(
            "Root-only preflight failed. Missing required files:\n"
            + "\n".join(missing)
        )

    if OUTPUT_ROOT.exists():
        raise FileExistsError(f"Refusing to overwrite an earlier run: {OUTPUT_ROOT}")
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=False)

    print("=" * 88)
    print(f"[{_ts()}] ROOT-NODE-ONLY sklearn experiment")
    print(f"[{_ts()}] 4 designs / 12 fitted model results")
    print(f"[{_ts()}] Same B&B set, split, features, hyperparameters")
    print(f"[{_ts()}] No dynamic features; no B&B target")
    print("=" * 88)

    bb_combined, bb_synthetic = load_bb_eligible_keys()

    combined_keys = set(bb_combined["_key"])
    synthetic_keys = set(bb_synthetic["_key"])

    combined_root = compute_root_targets(
        read_any(COMBINED_SUMMARY),
        "combined",
        combined_keys,
    )

    synthetic_root = compute_root_targets(
        read_any(SYNTHETIC_SUMMARY),
        "synthetic",
        synthetic_keys,
    )

    combined_feat, combined_cols = load_static_features(
        "combined",
        combined_keys,
    )

    synthetic_feat, synthetic_cols = load_static_features(
        "synthetic",
        synthetic_keys,
    )

    if combined_cols != synthetic_cols:
        raise RuntimeError(
            "Combined and Synthetic' static feature schemas differ.\n"
            f"Combined={combined_cols}\nSynthetic={synthetic_cols}"
        )

    combined_train_keys, combined_test_keys, split_manifest = (
        load_same_combined_split(combined_keys)
    )

    # Save root targets and reused split for audit.
    combined_root.to_csv(
        OUTPUT_ROOT / "root_targets_combined.csv",
        index=False,
    )

    synthetic_root.to_csv(
        OUTPUT_ROOT / "root_targets_synthetic_prime.csv",
        index=False,
    )

    split_manifest.to_csv(
        OUTPUT_ROOT / "reused_bb_combined_split_manifest.csv",
        index=False,
    )

    run_plan = [
        (1, "combined_to_combined", "classification"),
        (2, "combined_to_combined", "regression"),
        (3, "synthetic_to_combined", "classification"),
        (4, "synthetic_to_combined", "regression"),
    ]

    pd.DataFrame(
        [
            {
                "design_id": did,
                "train_setting": setting,
                "task": task,
                "input": "64_static_DI",
                "target": "root_LP_vs_SDP_PRIME",
            }
            for did, setting, task in run_plan
        ]
    ).to_csv(
        EXPERIMENT_MANIFEST_FILE,
        index=False,
    )

    all_results = []

    for design_id, setting, task in run_plan:
        rows = run_design(
            design_id=design_id,
            train_setting=setting,
            task=task,
            combined_root=combined_root,
            synthetic_root=synthetic_root,
            combined_features=combined_feat,
            synthetic_features=synthetic_feat,
            feature_cols=combined_cols,
            combined_train_keys=combined_train_keys,
            combined_test_keys=combined_test_keys,
        )

        all_results.extend(rows)

        pd.DataFrame(all_results).to_csv(
            MASTER_RESULTS_FILE,
            index=False,
        )

    results = pd.DataFrame(all_results)
    results.to_csv(MASTER_RESULTS_FILE, index=False)

    # Compact 4-design summary: best accuracy / rAccuracy per design,
    # while all 12 fitted-model rows remain in MASTER_RESULTS_FILE.
    summary_rows = []

    for design_id in [1, 2, 3, 4]:
        sub = results.loc[results["design_id"] == design_id].copy()

        if sub.empty:
            continue

        task = sub["task"].iloc[0]

        if task == "classification":
            best_idx = sub["accuracy"].astype(float).idxmax()
            best = sub.loc[best_idx]

            summary_rows.append(
                {
                    "design_id": design_id,
                    "train_test": best["train_test"],
                    "task": task,
                    "best_model": best["model"],
                    "primary_metric": "accuracy",
                    "primary_value": best["accuracy"],
                }
            )
        else:
            best_idx = sub["r_accuracy_paper"].astype(float).idxmax()
            best = sub.loc[best_idx]

            summary_rows.append(
                {
                    "design_id": design_id,
                    "train_test": best["train_test"],
                    "task": task,
                    "best_model": best["model"],
                    "primary_metric": "r_accuracy_paper",
                    "primary_value": best["r_accuracy_paper"],
                }
            )

    pd.DataFrame(summary_rows).to_csv(
        DESIGN_RESULTS_FILE,
        index=False,
    )

    print("\n" + "=" * 88)
    print(f"[{_ts()}] ROOT-ONLY EXPERIMENT COMPLETE")
    print(f"[{_ts()}] 4 designs / 12 fitted model results")
    print(f"[{_ts()}] Results -> {MASTER_RESULTS_FILE}")
    print(f"[{_ts()}] Root targets -> {OUTPUT_ROOT}")
    print("=" * 88)


if __name__ == "__main__":
    main()


[18:34:21] ROOT-NODE-ONLY sklearn experiment
[18:34:21] 4 designs / 12 fitted model results
[18:34:21] Same B&B set, split, features, hyperparameters
[18:34:21] No dynamic features; no B&B target
[18:34:21] [Synthetic' final B&B target] matching on instance_key; overlap=6069
[18:34:21] Locked B&B eligible sets: Combined=280, Synthetic'=6069


RuntimeError: combined summary: cannot find a definitively FULL-SDP root objective. Columns=['source', 'instance_key', 'instance_index', 'instance_id', 'instance_name', 'n', 'm', 'm_total', 'm_constr', 'm_original', 'm_solver', 'probtype', 'integrality_relaxed', 'convex', 'original_bounds_exist', 'budget_mode', 'lp_node_budget', 'sdp_node_budget', 'shared_upper_bound', 'lp_root_obj', 'sdp_root_obj', 'lp_root_time', 'sdp_root_time', 'root_winner', 'lp_leaf_bound_min', 'sdp_leaf_bound_min', 'lp_global_frontier_lb', 'sdp_global_frontier_lb', 'leaf_winner', 'lp_node_count', 'sdp_node_count', 'lp_max_depth', 'sdp_max_depth', 'lp_total_time', 'sdp_total_time', 'time_ratio_sdp_prime_over_lp', 'lp_first_infeasible_node', 'sdp_first_infeasible_node', 'lp_pruned_by_bound_count', 'sdp_pruned_by_bound_count', 'lp_gurobi_status_4_count', 'lp_final_queue_count', 'sdp_final_queue_count', '_key']. IMPORTANT: the original summary's sdp_root_obj denotes SDP-prime and MUST NOT be reused. Either add a ful

In [7]:
"""Eight root-node designs using identical eligible populations and splits as original.
No solver, no branch-and-bound is run.
"""
from __future__ import annotations

from datetime import datetime
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Tuple

import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import (
    GradientBoostingClassifier,
    GradientBoostingRegressor,
    RandomForestClassifier,
    RandomForestRegressor,
)
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

try:
    from xgboost import XGBClassifier, XGBRegressor
except Exception as e:
    raise ImportError(
        "xgboost is required because the final B&B experiment used XGB-C/XGB-R."
    ) from e


# ============================================================
# CONFIG — LOCKED TO FINAL B&B EXPERIMENT
# ============================================================

ROOT = Path(r"C:\Users\suuser\Desktop\minlplib")
FINAL_DIR = ROOT / "Areview" / "combined_minlplib_qplib"
DIM_TAG = "combined_minlplib_qplib"
SYNTHETIC_FINAL_DIR = (
    ROOT / "paperexperiments" / "synthetic_prime_minimal_revision"
    / "bb_separate_trees" / "synthetic_prime_8016_fixed100_v1"
    / "postrun_clean"
)

SKLEARN_RUN_NAME = "sklearn_all_runs_synthetic_prime_8016_20261021_014902_034048"


RANDOM_STATE = 42
TARGET_COL = "delta_root"
CLASS_COL = "y_root"

TAU = 0.01
ETA = 1e-3

USE_SAMPLE_WEIGHTS = False
WEIGHT_ALPHA = 0.30

EXPECTED_STATIC_FEATURES = 64
EXPECTED_COMBINED = 280
EXPECTED_SYNTHETIC = 6069

EXCLUDE_COMBINED_KEYS = {
    "QPLIB||QPLIB_2512",
    "QPLIB||QPLIB_3402",
    "QPLIB||QPLIB_3703",
}

# Use the same final B&B target files ONLY to recover the exact eligible
# instance sets. Their B&B delta/y values are NOT used as root targets.
BB_TARGET_PROBE = 10

COMBINED_BB_TARGET = (
    FINAL_DIR
    / "targets"
    / "method2"
    / DIM_TAG
    / f"targets_method2_relaxation_{DIM_TAG}_probe{BB_TARGET_PROBE}.csv"
)

SYNTHETIC_BB_TARGET = (SYNTHETIC_FINAL_DIR / "targets" / "method2"
                       / "synthetic_prime" / f"targets_method2_relaxation_synthetic_prime_probe{BB_TARGET_PROBE}.csv")

# Same static feature files as final B&B static runs.
COMBINED_STATIC = (
    FINAL_DIR
    / "combined_minlplib_qplib_features_DI_64_clean.xlsx"
)

SYNTHETIC_STATIC = SYNTHETIC_FINAL_DIR / "synthetic_prime_DI_64_clean.xlsx"

# Root objectives stored in the clean summary files.
COMBINED_SUMMARY = (
    FINAL_DIR
    / "summary_combined_minlplib_qplib_BB_clean.csv"
)

SYNTHETIC_SUMMARY = SYNTHETIC_FINAL_DIR / "synthetic_prime_BB_summary.csv"

# SAME exact Combined split generated by the final B&B sklearn runner.
BB_OUTPUT_ROOT = FINAL_DIR / SKLEARN_RUN_NAME

COMBINED_SPLIT_FILE = (
    BB_OUTPUT_ROOT
    / "combined_train_test_split_manifest.csv"
)

RUN_STAMP = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
OUTPUT_ROOT = FINAL_DIR / f"sklearn_root_8_designs_3_objectives_{RUN_STAMP}"

MASTER_RESULTS_FILE = (
    OUTPUT_ROOT
    / "all_root_only_model_results.csv"
)

DESIGN_RESULTS_FILE = (
    OUTPUT_ROOT
    / "root_only_design_summary.csv"
)

EXPERIMENT_MANIFEST_FILE = (
    OUTPUT_ROOT
    / "experiment_manifest.csv"
)


# ============================================================
# HELPERS
# ============================================================

def _ts():
    return datetime.now().strftime("%H:%M:%S")


def read_any(path: Path) -> pd.DataFrame:
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f"Required file not found:\n{path}")

    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)

    if path.suffix.lower() in {".xlsx", ".xls"}:
        return pd.read_excel(path)

    raise ValueError(f"Unsupported file type: {path}")


def _canon_name(x) -> str:
    s = str(x).strip()
    if s.lower().endswith(".pkl"):
        s = s[:-4]
    return s


def _norm_source(x) -> str:
    return str(x).strip().upper()


def _require_columns(df: pd.DataFrame, cols: Iterable[str], label: str):
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ValueError(
            f"{label} missing columns {missing}. "
            f"Available columns: {list(df.columns)}"
        )


def _assert_unique(df: pd.DataFrame, key: str, label: str):
    if df[key].duplicated().any():
        ex = (
            df.loc[df[key].duplicated(keep=False), key]
            .astype(str)
            .head(20)
            .tolist()
        )
        raise ValueError(f"{label}: duplicate {key}. Examples: {ex}")


# ============================================================
# KEY HANDLING — SAME CONVENTIONS AS FINAL B&B SCRIPT
# ============================================================

def canonicalize_combined_key(df: pd.DataFrame, label: str) -> pd.DataFrame:
    out = df.copy()

    if "instance_key" in out.columns:
        out["_key"] = out["instance_key"].astype(str).str.strip()
    else:
        if "instance_name" not in out.columns:
            raise ValueError(
                f"{label}: need instance_key or instance_name."
            )

        out["instance_name"] = out["instance_name"].map(_canon_name)

        if "source" in out.columns:
            src = out["source"].map(_norm_source)

        elif "id" in out.columns:
            ids = out["id"].astype(str).str.strip()

            src = pd.Series(
                np.where(
                    ids.str.startswith("MINLPLIB__"),
                    "MINLPLIB",
                    np.where(
                        ids.str.startswith("QPLIB__"),
                        "QPLIB",
                        "",
                    ),
                ),
                index=out.index,
                dtype=object,
            )

            if (src == "").any():
                bad = out.loc[src == "", "id"].head(20).tolist()
                raise ValueError(
                    f"{label}: cannot infer source from id. Examples: {bad}"
                )
        else:
            raise ValueError(f"{label}: cannot construct Combined key.")

        out["_key"] = (
            src.astype(str)
            + "||"
            + out["instance_name"].astype(str)
        )

    out["_key"] = out["_key"].astype(str).str.strip()

    if "source" not in out.columns:
        out["source"] = out["_key"].str.split("||", n=1).str[0]

    out["source"] = out["source"].map(_norm_source)

    if "instance_name" not in out.columns:
        out["instance_name"] = out["_key"].str.split("||", n=1).str[1]

    out["instance_name"] = out["instance_name"].map(_canon_name)

    _assert_unique(out, "_key", label)
    return out


def canonicalize_synthetic_key(
    df: pd.DataFrame, label: str, target_keys: Optional[set] = None
) -> pd.DataFrame:
    """Canonical SYNTHETIC||<original-id>; never join by row positions."""
    out = df.copy()
    candidates = [c for c in ("instance_key", "dataset_id", "instance_id", "id")
                  if c in out.columns]
    if not candidates:
        raise ValueError(f"{label}: no original numeric ID or instance_key")
    best_col, best_keys, best_overlap = None, None, -1
    for col in candidates:
        raw = out[col].astype("string").str.strip()
        raw = raw.str.replace(r"^SYNTHETIC\|\|", "", regex=True)
        if raw.isna().any() or not raw.str.fullmatch(r"[0-9]+").all():
            continue
        keys = "SYNTHETIC||" + raw.astype(str)
        overlap = len(set(keys) & target_keys) if target_keys is not None else len(keys)
        if overlap > best_overlap:
            best_col, best_keys, best_overlap = col, keys, overlap
    if best_keys is None or (target_keys is not None and best_overlap == 0):
        raise ValueError(f"{label}: cannot align original Synthetic' IDs to targets")
    out["_key"] = best_keys
    _assert_unique(out, "_key", label)
    print(f"[{_ts()}] [{label}] matching on {best_col}; overlap={best_overlap}")
    return out


# ============================================================
# EXACT B&B ELIGIBLE INSTANCE SETS
# ============================================================

def load_bb_eligible_keys() -> Tuple[pd.DataFrame, pd.DataFrame]:
    combined = canonicalize_combined_key(
        read_any(COMBINED_BB_TARGET),
        "Combined final B&B target",
    )

    synthetic = canonicalize_synthetic_key(
        read_any(SYNTHETIC_BB_TARGET),
        "Synthetic' final B&B target",
    )

    # Same explicit three Combined exclusions as the final B&B experiment.
    combined = (
        combined.loc[
            ~combined["_key"].isin(EXCLUDE_COMBINED_KEYS)
        ]
        .copy()
        .reset_index(drop=True)
    )

    if len(combined) != EXPECTED_COMBINED:
        raise RuntimeError(
            f"Expected exactly {EXPECTED_COMBINED} final Combined B&B instances, "
            f"found {len(combined)}."
        )

    if len(synthetic) != EXPECTED_SYNTHETIC:
        raise RuntimeError(
            f"Expected exactly {EXPECTED_SYNTHETIC} final Synthetic' B&B instances, "
            f"found {len(synthetic)}."
        )

    _assert_unique(combined, "_key", "Combined B&B eligible set")
    _assert_unique(synthetic, "_key", "Synthetic' B&B eligible set")

    print(
        f"[{_ts()}] Locked B&B eligible sets: "
        f"Combined={len(combined)}, Synthetic'={len(synthetic)}",
        flush=True,
    )

    return combined, synthetic


# ============================================================
# ROOT TARGETS
# ============================================================

# ============================================================
# THREE ROOT OBJECTIVES — FROM ROOT SOLVE OUTPUT FILES, NOT B&B LABELS
# ============================================================
# Input tables may have extra instances. Eligible IDs stay locked to the original
# B&B experiment. We never determine eligibility from the objective tables.
# Set explicit paths below if automatic discovery cannot unambiguously find them.
# Authoritative ROOT SOLVE result tables: never scan unrelated archives or B&B targets.
QPLIB_ROOT_FILES = [ROOT / "Areview" / "qplib" / "qplib_root_results_ALL143.csv"]
MINLPLIB_ROOT_FILES = [ROOT / "minlplib_objectives_final.csv"]
SYNTHETIC_ROOT_FILES = [ROOT / "paperexperiments" / "synthetic_prime_minimal_revision" / "instances_mixed_8016_root_results.csv"]

LP_COLUMNS = ("lp_obj",)
SDP_COLUMNS = ("sdp_obj",)
SDPP_COLUMNS = ("sdpp_obj", "sdp_prime_obj")


def _status_and_objective(df, prefix, value_col, alias_status):
    values = pd.to_numeric(df[value_col], errors="coerce").to_numpy(dtype=float)
    status_col = next((c for c in alias_status if c in df.columns), None)
    if status_col is None:
        statuses = np.where(np.isneginf(values), "unbounded", np.where(np.isfinite(values), "optimal_unverified", "missing"))
    else:
        statuses = df[status_col].astype(str).str.strip().str.lower().to_numpy()
        # A stored -inf is a mathematical unbounded result only when status agrees.
        for i, (v, st) in enumerate(zip(values, statuses)):
            if np.isneginf(v) and st not in ("unbounded", "dual_infeasible", "primal_unbounded"):
                raise RuntimeError(f"{prefix}: -inf objective but status {st!r} at row {i}")
            if np.isfinite(v) and st not in ("optimal", "optimal_unverified"):
                values[i] = np.nan  # only trust solved finite objectives
            if np.isposinf(v):
                values[i] = np.nan
    return values, statuses


def _canonical_result_keys(df, origin):
    if origin == "minlplib":
        if "source_pkl" in df.columns:
            # e.g. 'ex3_1_3.pkl', unlike 'instance_name' = 'ex3_1_3_23'.
            names = df["source_pkl"].map(lambda v: Path(str(v)).stem)
        elif "id_global" in df.columns:
            names = df["id_global"].astype(str).str.split("__", regex=False).str[0]
        else:
            raise RuntimeError("MINLPLib objective CSV needs source_pkl or id_global for identity")
        return "MINLPLIB||" + names.astype(str)
    if origin == "qplib":
        if "name" not in df.columns:
            raise RuntimeError("QPLIB root CSV needs name")
        return "QPLIB||" + df["name"].astype(str).str.strip().map(_canon_name)
    if origin == "synthetic":
        if "id" not in df.columns:
            raise RuntimeError("Synthetic' root CSV needs original numeric id")
        ids = pd.to_numeric(df["id"], errors="coerce")
        if ids.isna().any() or (ids != np.floor(ids)).any():
            raise RuntimeError("Synthetic' root id must be integral")
        return "SYNTHETIC||" + ids.astype("int64").astype(str)
    raise ValueError(origin)


def _read_authoritative(path, origin, eligible):
    df = read_any(path).copy()
    df["_key"] = _canonical_result_keys(df, origin)
    df = df.loc[df["_key"].isin(eligible)].copy().reset_index(drop=True)
    if df["_key"].duplicated().any():
        raise RuntimeError(f"{path}: repeated eligible instance keys: {df.loc[df['_key'].duplicated(), '_key'].head(10).tolist()}")
    output = pd.DataFrame({"_key": df["_key"]})
    col_options = {"lp": LP_COLUMNS, "sdp": SDP_COLUMNS, "sdpp": SDPP_COLUMNS}
    for method, aliases in col_options.items():
        col = next((name for name in aliases if name in df.columns), None)
        if col is None:
            raise RuntimeError(f"{path}: missing {method} objective column; columns: {list(df.columns)}")
        values, statuses = _status_and_objective(
            df, f"{origin} {method}", col,
            [f"{method}_status", "sdp_prime_status" if method=="sdpp" else f"{method}_status"]
        )
        output[f"{method}_obj"] = values
        output[f"{method}_status"] = statuses
        output[f"{method}_source"] = str(path)
    return output


def load_three_root_objectives(kind, eligible):
    if kind == "synthetic":
        sources = [(SYNTHETIC_ROOT_FILES[0], "synthetic")]
    else:
        sources = [(MINLPLIB_ROOT_FILES[0], "minlplib"), (QPLIB_ROOT_FILES[0], "qplib")]
    frames = [_read_authoritative(path, origin, eligible) for path, origin in sources]
    assembled = pd.concat(frames, ignore_index=True)
    _assert_unique(assembled, "_key", f"{kind} authoritative root sources")
    complete_keys = set(assembled["_key"])
    absent = eligible - complete_keys
    if absent:
        audit = OUTPUT_ROOT / f"MISSING_{kind.upper()}_ROOT_OBJECTIVES.csv"
        pd.DataFrame({"_key": sorted(absent)}).to_csv(audit, index=False)
        raise RuntimeError(f"{kind}: {len(absent)} locked instances have no objective record; audit {audit}. First: {sorted(absent)[:15]}")
    assembled = assembled.set_index("_key").loc[sorted(eligible)].reset_index()
    report = OUTPUT_ROOT / f"root_objective_status_audit_{kind}.csv"
    assembled.to_csv(report, index=False)
    for method in ("lp", "sdp", "sdpp"):
        vals = assembled[f"{method}_obj"].to_numpy(float)
        print(f"[{_ts()}] {kind} {method}: finite={np.isfinite(vals).sum()}, unbounded_below={np.isneginf(vals).sum()}, other_missing={np.isnan(vals).sum()}", flush=True)
    # LP and SDP-prime need finite objectives for the two experiments.
    # An SDP=-inf value is valid and receives delta=+1 by a declared convention.
    for method in ("lp", "sdpp"):
        invalid = ~np.isfinite(assembled[f"{method}_obj"].to_numpy(float))
        if invalid.any():
            bad = assembled.loc[invalid, ["_key", f"{method}_obj", f"{method}_status"]]
            raise RuntimeError(f"{kind}: {len(bad)} locked instances do not have finite {method} objectives. No rows removed. Audit {report}.\n{bad.head(20).to_string(index=False)}")
    sdp = assembled["sdp_obj"].to_numpy(float)
    bad = ~(np.isfinite(sdp) | np.isneginf(sdp))
    if bad.any():
        rows = assembled.loc[bad, ["_key", "sdp_obj", "sdp_status"]]
        raise RuntimeError(f"{kind}: {len(rows)} SDP outcomes neither finite nor unbounded below. No rows removed. Audit {report}.\n{rows.head(20).to_string(index=False)}")
    return assembled

def compute_root_targets(root_frame, dataset_kind, eligible_keys, comparison):
    if set(root_frame['_key']) != eligible_keys or len(root_frame)!=len(eligible_keys):
        raise RuntimeError(f"{dataset_kind}: root objectives do not exactly match locked eligible IDs")
    out=root_frame.copy()
    method='sdp' if comparison=='LP_vs_SDP' else 'sdpp'
    zlp=out['lp_obj'].to_numpy(float)
    zother=out[f'{method}_obj'].to_numpy(float)
    if not np.isfinite(zlp).all():
        raise RuntimeError(f"{dataset_kind}: nonfinite LP objective")
    if comparison=='LP_vs_SDP_PRIME' and not np.isfinite(zother).all():
        raise RuntimeError(f"{dataset_kind}: nonfinite SDP-prime objective")
    if comparison=='LP_vs_SDP' and not (np.isfinite(zother) | np.isneginf(zother)).all():
        raise RuntimeError(f"{dataset_kind}: invalid SDP statuses: check root audit")
    out['lp_root_obj']=zlp
    out['sdp_root_obj']=zother # compatibility with original model runner
    # Mathematical limit of (LP - SDP)/(|LP|+|SDP|+ETA) as SDP -> -infinity is +1.
    # Explicitly record the unbounded cases; do not treat them as missing or drop.
    unbounded=np.isneginf(zother)
    delta=np.empty(len(zlp), dtype=float)
    delta[unbounded]=1.0
    finite=~unbounded
    delta[finite]=(zlp[finite]-zother[finite])/(np.abs(zlp[finite])+np.abs(zother[finite])+ETA)
    out['sdp_unbounded_below']=unbounded if comparison=='LP_vs_SDP' else False
    out[TARGET_COL]=delta
    out[CLASS_COL]=(out[TARGET_COL].to_numpy(float)>-TAU).astype(int)
    out['source']= 'SYNTHETIC' if dataset_kind=='synthetic' else out['_key'].str.split('||',regex=False).str[0]
    out['instance_name']=out['_key'].str.split('||',regex=False).str[-1]
    out['root_winner_thresholded']=np.where(out[CLASS_COL]==1,'LP',method.upper())
    out['comparison']=comparison
    print(f"[{_ts()}] {dataset_kind} {comparison}: labels {out[CLASS_COL].value_counts().sort_index().to_dict()}")
    return out


# ============================================================
# STATIC FEATURE LOADING — EXACT SAME 64 FEATURES
# ============================================================

FORBIDDEN_STATIC_COLS = {
    "id",
    "id_key",
    "id_global",
    "instance_id",
    "instanceid",
    "instance_index",
    "instance_name",
    "instance_key",
    "problem_name",
    "name",
    "source",
    "_key",
    "y",
    "winner",
    "delta",
    "y_root",
    "delta_root",
    "lp_obj",
    "sdp_prime_obj",
    "sdp_obj",
    "sdpp_obj",
    "lp_root_obj",
    "sdp_root_obj",
    "lp_status",
    "sdp_status",
    "sdp_prime_status",
    "sdpp_status",
}


def load_static_features(
    dataset_kind: str,
    eligible_keys: set,
) -> Tuple[pd.DataFrame, List[str]]:
    path = (
        COMBINED_STATIC
        if dataset_kind == "combined"
        else SYNTHETIC_STATIC
    )

    raw = read_any(path)

    if dataset_kind == "combined":
        df = canonicalize_combined_key(
            raw,
            "Combined static features",
        )
    else:
        df = canonicalize_synthetic_key(
            raw,
            "Synthetic' static features",
            target_keys=eligible_keys,
        )

    feature_cols = [
        c
        for c in df.columns
        if c not in FORBIDDEN_STATIC_COLS
    ]

    if len(feature_cols) != EXPECTED_STATIC_FEATURES:
        raise RuntimeError(
            f"{dataset_kind}: expected exactly {EXPECTED_STATIC_FEATURES} "
            f"static DI features, found {len(feature_cols)}.\n"
            f"Columns: {feature_cols}"
        )

    missing = eligible_keys - set(df["_key"])
    if missing:
        raise RuntimeError(
            f"{dataset_kind}: static feature file misses B&B-eligible instances. "
            f"Examples: {sorted(missing)[:30]}"
        )

    out = (
        df.loc[df["_key"].isin(eligible_keys), ["_key"] + feature_cols]
        .copy()
        .reset_index(drop=True)
    )

    for c in feature_cols:
        out[c] = pd.to_numeric(out[c], errors="coerce")

    _assert_unique(out, "_key", f"{dataset_kind} static features")

    return out, feature_cols


# ============================================================
# SAME B&B COMBINED SPLIT
# ============================================================

def load_same_combined_split(
    combined_keys: set,
) -> Tuple[set, set, pd.DataFrame]:
    m = read_any(COMBINED_SPLIT_FILE)

    _require_columns(
        m,
        ["_key", "split"],
        "final B&B Combined split manifest",
    )

    m["_key"] = m["_key"].astype(str).str.strip()
    m["split"] = m["split"].astype(str).str.strip().str.lower()

    _assert_unique(m, "_key", "final B&B Combined split manifest")

    if set(m["_key"]) != combined_keys:
        raise RuntimeError(
            "The B&B split manifest does not match the final 280-instance "
            "Combined set."
        )

    if set(m["split"]) - {"train", "test"}:
        raise RuntimeError(
            "B&B split manifest contains values other than train/test."
        )

    train_keys = set(m.loc[m["split"] == "train", "_key"])
    test_keys = set(m.loc[m["split"] == "test", "_key"])

    if train_keys & test_keys:
        raise RuntimeError("B&B split manifest has train/test overlap.")

    print(
        f"[{_ts()}] Reusing exact B&B Combined split: "
        f"train={len(train_keys)}, test={len(test_keys)}",
        flush=True,
    )

    return train_keys, test_keys, m


# ============================================================
# MODELS — EXACT SAME HYPERPARAMETERS AS FINAL B&B RUNNER
# ============================================================

def make_models(task: str) -> Dict[str, object]:
    if task == "classification":
        return {
            "XGB-C": XGBClassifier(
                n_estimators=400,
                learning_rate=0.08,
                max_depth=6,
                subsample=0.8,
                colsample_bytree=0.8,
                reg_alpha=0.1,
                reg_lambda=1.0,
                random_state=RANDOM_STATE,
                n_jobs=-1,
                eval_metric="logloss",
            ),
            "GBC": GradientBoostingClassifier(
                random_state=RANDOM_STATE,
            ),
            "RFC": RandomForestClassifier(
                n_estimators=400,
                max_features="sqrt",
                class_weight="balanced",
                n_jobs=-1,
                random_state=RANDOM_STATE,
            ),
        }

    if task == "regression":
        return {
            "XGB-R": XGBRegressor(
                n_estimators=400,
                learning_rate=0.08,
                max_depth=6,
                subsample=0.8,
                colsample_bytree=0.8,
                reg_alpha=0.1,
                reg_lambda=1.0,
                random_state=RANDOM_STATE,
                n_jobs=-1,
            ),
            "GBR": GradientBoostingRegressor(
                random_state=RANDOM_STATE,
            ),
            "RFR": RandomForestRegressor(
                n_estimators=400,
                n_jobs=-1,
                random_state=RANDOM_STATE,
            ),
        }

    raise ValueError(f"Unknown task={task}")


def build_pipeline(model) -> Pipeline:
    try:
        imputer = SimpleImputer(
            strategy="median",
            keep_empty_features=True,
        )
    except TypeError:
        imputer = SimpleImputer(
            strategy="median",
        )

    return Pipeline(
        [
            ("imputer", imputer),
            ("model", model),
        ]
    )


# ============================================================
# METRICS — SAME AS FINAL B&B RUNNER
# ============================================================

def safe_pearson_r(y_true, y_pred):
    yt = np.asarray(y_true, dtype=float)
    yp = np.asarray(y_pred, dtype=float)

    mask = np.isfinite(yt) & np.isfinite(yp)
    yt = yt[mask]
    yp = yp[mask]

    if len(yt) < 2 or np.std(yt) <= 0 or np.std(yp) <= 0:
        return float("nan")

    r = np.corrcoef(yt, yp)[0, 1]
    return float(r) if np.isfinite(r) else float("nan")


def r_accuracy_paper(y_true_delta, y_pred_delta, eps=TAU):
    yt = np.asarray(y_true_delta, dtype=float)
    yp = np.asarray(y_pred_delta, dtype=float)

    mask = np.isfinite(yt) & np.isfinite(yp)
    yt = yt[mask]
    yp = yp[mask]

    if len(yt) == 0:
        return float("nan")

    same_sign = np.sign(yt) == np.sign(yp)

    forgiveness = (
        (yt > -eps)
        & (yt < 0)
        & (yp > 0)
    )

    return float(np.mean(same_sign | forgiveness))


def r_accuracy_tol(y_true_delta, y_pred_delta, eps=TAU):
    yt = np.asarray(y_true_delta, dtype=float)
    yp = np.asarray(y_pred_delta, dtype=float)

    mask = np.isfinite(yt) & np.isfinite(yp)
    yt = yt[mask]
    yp = yp[mask]

    if len(yt) == 0:
        return float("nan")

    same = np.sign(yt) == np.sign(yp)

    q2 = (
        (yt > -eps)
        & (yt < 0)
        & (yp > 0)
    )

    q4 = (
        (yt > 0)
        & (yt < eps)
        & (yp < 0)
        & (yp > -eps)
    )

    return float(np.mean(same | q2 | q4))


def decision_from_delta(delta_values):
    return (
        np.asarray(delta_values, dtype=float)
        > -TAU
    ).astype(int)


# ============================================================
# MERGE
# ============================================================

def merge_targets_features(
    target_df: pd.DataFrame,
    feature_df: pd.DataFrame,
    feature_cols: List[str],
    label: str,
) -> pd.DataFrame:
    missing = set(target_df["_key"]) - set(feature_df["_key"])

    if missing:
        raise RuntimeError(
            f"{label}: targets missing static features. "
            f"Examples: {sorted(missing)[:30]}"
        )

    target_cols = [
        c
        for c in [
            "_key",
            "source",
            "instance_index",
            "instance_id",
            "instance_name",
            "lp_root_obj",
            "sdp_root_obj",
            TARGET_COL,
            CLASS_COL,
        ]
        if c in target_df.columns
    ]

    out = target_df[target_cols].merge(
        feature_df[["_key"] + feature_cols],
        on="_key",
        how="left",
        validate="one_to_one",
    )

    if len(out) != len(target_df):
        raise RuntimeError(f"{label}: merge changed row count.")

    return out


# ============================================================
# ONE DESIGN
# ============================================================

def run_design(
    *,
    design_id: int,
    train_setting: str,
    task: str,
    combined_root: pd.DataFrame,
    comparison: str,
    synthetic_root: pd.DataFrame,
    combined_features: pd.DataFrame,
    synthetic_features: pd.DataFrame,
    feature_cols: List[str],
    combined_train_keys: set,
    combined_test_keys: set,
) -> List[dict]:
    if train_setting == "combined_to_combined":
        all_combined = merge_targets_features(
            combined_root,
            combined_features,
            feature_cols,
            "Combined root",
        )

        train_df = (
            all_combined.loc[
                all_combined["_key"].isin(combined_train_keys)
            ]
            .copy()
            .reset_index(drop=True)
        )

        test_df = (
            all_combined.loc[
                all_combined["_key"].isin(combined_test_keys)
            ]
            .copy()
            .reset_index(drop=True)
        )

        setting_label = "Combined→Combined"

    elif train_setting == "synthetic_to_combined":
        # IMPORTANT: exactly mirrors the final B&B sklearn runner.
        # Train on ALL Synthetic'; test on ALL 280 Combined.
        train_df = merge_targets_features(
            synthetic_root,
            synthetic_features,
            feature_cols,
            "Synthetic' root train",
        )

        test_df = merge_targets_features(
            combined_root,
            combined_features,
            feature_cols,
            "Combined root transfer test",
        )

        setting_label = "Synthetic′→Combined"

    else:
        raise ValueError(train_setting)

    if task == "classification":
        y_train = train_df[CLASS_COL].to_numpy(int)
        y_test = test_df[CLASS_COL].to_numpy(int)

        if len(np.unique(y_train)) < 2:
            raise RuntimeError(
                f"Design {design_id}: classification train set is single-class."
            )
    else:
        y_train = None
        y_test = test_df[CLASS_COL].to_numpy(int)

    d_train = train_df[TARGET_COL].to_numpy(float)
    d_test = test_df[TARGET_COL].to_numpy(float)

    X_train = train_df[feature_cols].replace([np.inf, -np.inf], np.nan)
    X_test = test_df[feature_cols].replace([np.inf, -np.inf], np.nan)

    run_dir = (
        OUTPUT_ROOT
        / f"design_{design_id:02d}__{comparison}__{train_setting}__{task}"
    )

    run_dir.mkdir(parents=True, exist_ok=True)

    split_out = pd.concat(
        [
            train_df[["_key", TARGET_COL, CLASS_COL]].assign(split="train"),
            test_df[["_key", TARGET_COL, CLASS_COL]].assign(split="test"),
        ],
        ignore_index=True,
    )

    split_out.to_csv(
        run_dir / "instance_split_root_targets.csv",
        index=False,
    )

    pd.DataFrame({"feature": feature_cols}).to_csv(
        run_dir / "feature_columns.csv",
        index=False,
    )

    print("\n" + "=" * 88)
    print(
        f"[{_ts()}] DESIGN {design_id} | {setting_label} | ROOT {task.upper()}"
    )
    print(
        f"[{_ts()}] train={len(train_df)} test={len(test_df)} "
        f"features={len(feature_cols)}"
    )
    print(
        f"[{_ts()}] root train labels="
        f"{train_df[CLASS_COL].value_counts().sort_index().to_dict()} | "
        f"test labels="
        f"{test_df[CLASS_COL].value_counts().sort_index().to_dict()}"
    )
    print("=" * 88)

    result_rows = []

    for model_name, model in make_models(task).items():
        print(f"[{_ts()}] training {model_name}...", flush=True)

        pipe = build_pipeline(model)

        if task == "classification":
            pipe.fit(X_train, y_train)

            pred = pipe.predict(X_test).astype(int)

            if hasattr(pipe, "predict_proba") and len(np.unique(y_test)) > 1:
                prob = pipe.predict_proba(X_test)[:, 1]
                auc = float(roc_auc_score(y_test, prob))
            else:
                prob = np.full(len(y_test), np.nan)
                auc = float("nan")

            acc = float(accuracy_score(y_test, pred))
            bal = float(balanced_accuracy_score(y_test, pred))
            f1_lp = float(f1_score(y_test, pred, zero_division=0))
            f1_macro = float(
                f1_score(
                    y_test,
                    pred,
                    average="macro",
                    zero_division=0,
                )
            )
            precision = float(
                precision_score(y_test, pred, zero_division=0)
            )
            recall = float(
                recall_score(y_test, pred, zero_division=0)
            )

            cm = confusion_matrix(y_test, pred, labels=[0, 1])

            pd.DataFrame(
                cm,
                index=[f"true_{comparison.split('_vs_')[-1]}(0)", "true_LP(1)"],
                columns=[f"pred_{comparison.split('_vs_')[-1]}(0)", "pred_LP(1)"],
            ).to_csv(
                run_dir / f"cm_{model_name}.csv",
                index=True,
            )

            disp = ConfusionMatrixDisplay(
                cm,
                display_labels=[comparison.split("_vs_")[-1], "LP"],
            )
            disp.plot()
            plt.title(
                f"Root | {setting_label} | {model_name}"
            )
            plt.tight_layout()
            plt.savefig(
                run_dir / f"cm_{model_name}.png",
                dpi=200,
            )
            plt.close()

            pred_df = test_df[
                ["_key", TARGET_COL, CLASS_COL]
            ].copy()

            pred_df["y_pred"] = pred
            pred_df["prob_LP"] = prob

            pred_df.to_csv(
                run_dir / f"preds_{model_name}.csv",
                index=False,
            )

            result = {
                "design_id": design_id,
                "comparison": comparison,
                "train_test": setting_label,
                "task": task,
                "input": "64_static_DI",
                "target": "root_" + comparison,
                "model": model_name,
                "n_train": len(train_df),
                "n_test": len(test_df),
                "accuracy": acc,
                "balanced_accuracy": bal,
                "f1_lp": f1_lp,
                "f1_macro": f1_macro,
                "precision_LP": precision,
                "recall_LP": recall,
                "roc_auc": auc,
                "rmse": np.nan,
                "mae": np.nan,
                "r2": np.nan,
                "pearson_r": np.nan,
                "r_accuracy_paper": np.nan,
                "r_accuracy_tol": np.nan,
                "decision_accuracy": np.nan,
            }

            print(
                f"[{_ts()}] {model_name}: "
                f"acc={acc:.4f} bal={bal:.4f} macroF1={f1_macro:.4f}",
                flush=True,
            )

        else:
            pipe.fit(X_train, d_train)

            delta_pred = pipe.predict(X_test)
            pred_class = decision_from_delta(delta_pred)

            mse = float(
                mean_squared_error(
                    d_test,
                    delta_pred,
                )
            )
            rmse = float(np.sqrt(mse))
            mae = float(
                mean_absolute_error(
                    d_test,
                    delta_pred,
                )
            )
            r2 = float(
                r2_score(
                    d_test,
                    delta_pred,
                )
            )
            pearson = safe_pearson_r(
                d_test,
                delta_pred,
            )
            r_paper = r_accuracy_paper(
                d_test,
                delta_pred,
            )
            r_tol = r_accuracy_tol(
                d_test,
                delta_pred,
            )
            decision_acc = float(
                accuracy_score(
                    y_test,
                    pred_class,
                )
            )

            pred_df = test_df[
                ["_key", TARGET_COL, CLASS_COL]
            ].copy()

            pred_df["delta_pred"] = delta_pred
            pred_df["y_pred_from_delta"] = pred_class

            pred_df.to_csv(
                run_dir / f"preds_{model_name}.csv",
                index=False,
            )

            result = {
                "design_id": design_id,
                "comparison": comparison,
                "train_test": setting_label,
                "task": task,
                "input": "64_static_DI",
                "target": "root_" + comparison,
                "model": model_name,
                "n_train": len(train_df),
                "n_test": len(test_df),
                "accuracy": np.nan,
                "balanced_accuracy": np.nan,
                "f1_lp": np.nan,
                "f1_macro": np.nan,
                "precision_LP": np.nan,
                "recall_LP": np.nan,
                "roc_auc": np.nan,
                "rmse": rmse,
                "mae": mae,
                "r2": r2,
                "pearson_r": pearson,
                "r_accuracy_paper": r_paper,
                "r_accuracy_tol": r_tol,
                "decision_accuracy": decision_acc,
            }

            print(
                f"[{_ts()}] {model_name}: "
                f"rAcc_paper={r_paper:.4f} "
                f"rAcc_tol={r_tol:.4f} "
                f"RMSE={rmse:.5f} MAE={mae:.5f} R2={r2:.4f}",
                flush=True,
            )

        joblib.dump(
            pipe,
            run_dir / f"model_{model_name}.joblib",
        )

        result_rows.append(result)

    pd.DataFrame(result_rows).to_csv(
        run_dir / "results_all_models.csv",
        index=False,
    )

    return result_rows


# ============================================================
# MAIN
# ============================================================

def main():
    required=[COMBINED_BB_TARGET,SYNTHETIC_BB_TARGET,COMBINED_STATIC,
              SYNTHETIC_STATIC,COMBINED_SPLIT_FILE]
    missing=[str(p) for p in required if not Path(p).exists()]
    if missing:
        raise FileNotFoundError("Missing locked experiment inputs:\n"+"\n".join(missing))
    if OUTPUT_ROOT.exists():
        raise FileExistsError(f"Refusing to overwrite: {OUTPUT_ROOT}")
    OUTPUT_ROOT.mkdir(parents=True,exist_ok=False)
    print("ROOT NODE ONLY: 8 DESIGNS / 24 MODELS; NO B&B SOLVING",flush=True)
    bb_combined,bb_synthetic=load_bb_eligible_keys()
    combined_keys=set(bb_combined['_key'])
    synthetic_keys=set(bb_synthetic['_key'])
    combined_feat,combined_cols=load_static_features('combined',combined_keys)
    synthetic_feat,synthetic_cols=load_static_features('synthetic',synthetic_keys)
    if combined_cols!=synthetic_cols:
        raise RuntimeError('Different 64-feature schemas between Combined and Synthetic')
    combined_train_keys,combined_test_keys,split_manifest=load_same_combined_split(combined_keys)
    if len(combined_train_keys)!=177 or len(combined_test_keys)!=103:
        raise RuntimeError('Train/test counts changed from original 177/103')
    split_manifest.to_csv(OUTPUT_ROOT/'reused_bb_combined_split_manifest.csv',index=False)

    print('Loading PRECOMPUTED root objective files only (no QCQP solving)',flush=True)
    combined_objectives=load_three_root_objectives('combined',combined_keys)
    synthetic_objectives=load_three_root_objectives('synthetic',synthetic_keys)
    combined_objectives.to_csv(OUTPUT_ROOT/'root_objective_sources_combined.csv',index=False)
    synthetic_objectives.to_csv(OUTPUT_ROOT/'root_objective_sources_synthetic.csv',index=False)

    plan=[]
    for comp in ('LP_vs_SDP','LP_vs_SDP_PRIME'):
        for setting,task in (("combined_to_combined","classification"),
                             ("combined_to_combined","regression"),
                             ("synthetic_to_combined","classification"),
                             ("synthetic_to_combined","regression")):
            plan.append((len(plan)+1,comp,setting,task))
    pd.DataFrame(plan,columns=['design_id','comparison','train_setting','task']).to_csv(
        EXPERIMENT_MANIFEST_FILE,index=False)
    all_results=[]
    for did,comp,setting,task in plan:
        combined_root=compute_root_targets(combined_objectives,'combined',combined_keys,comp)
        synthetic_root=compute_root_targets(synthetic_objectives,'synthetic',synthetic_keys,comp)
        combined_root.to_csv(OUTPUT_ROOT/f'root_targets_combined_{comp}.csv',index=False)
        synthetic_root.to_csv(OUTPUT_ROOT/f'root_targets_synthetic_{comp}.csv',index=False)
        rows=run_design(design_id=did,comparison=comp,train_setting=setting,task=task,
                        combined_root=combined_root,synthetic_root=synthetic_root,
                        combined_features=combined_feat,synthetic_features=synthetic_feat,
                        feature_cols=combined_cols,combined_train_keys=combined_train_keys,
                        combined_test_keys=combined_test_keys)
        all_results.extend(rows)
        pd.DataFrame(all_results).to_csv(MASTER_RESULTS_FILE,index=False)
    df=pd.DataFrame(all_results)
    if len(df)!=24:
        raise RuntimeError(f'Expected 24 model results; got {len(df)}')
    summary=[]
    for did,group in df.groupby('design_id',sort=True):
        metric='accuracy' if group['task'].iloc[0]=='classification' else 'r_accuracy_paper'
        best=group.loc[group[metric].astype(float).idxmax()]
        summary.append(dict(design_id=did,comparison=best['comparison'],
                            train_test=best['train_test'],task=best['task'],
                            best_model=best['model'],primary_metric=metric,primary_value=best[metric]))
    pd.DataFrame(summary).to_csv(DESIGN_RESULTS_FILE,index=False)
    print(f"COMPLETE: 8 designs, 24 models. Results -> {MASTER_RESULTS_FILE}",flush=True)

if __name__=='__main__':
    main()


ROOT NODE ONLY: 8 DESIGNS / 24 MODELS; NO B&B SOLVING
[19:56:35] [Synthetic' final B&B target] matching on instance_key; overlap=6069
[19:56:35] Locked B&B eligible sets: Combined=280, Synthetic'=6069
[19:56:39] [Synthetic' static features] matching on id; overlap=6069
[19:56:39] Reusing exact B&B Combined split: train=177, test=103
Loading PRECOMPUTED root objective files only (no QCQP solving)
[19:56:39] combined lp: finite=280, unbounded_below=0, other_missing=0
[19:56:39] combined sdp: finite=209, unbounded_below=69, other_missing=2
[19:56:39] combined sdpp: finite=280, unbounded_below=0, other_missing=0


RuntimeError: combined: 2 SDP outcomes neither finite nor unbounded below. No rows removed. Audit C:\Users\suuser\Desktop\minlplib\Areview\combined_minlplib_qplib\sklearn_root_8_designs_3_objectives_20261022_195635_535240\root_objective_status_audit_combined.csv.
             _key  sdp_obj sdp_status
MINLPLIB||ex5_3_3      NaN    unknown
QPLIB||QPLIB_9048      NaN    unknown

In [10]:
"""Eight root-node designs using identical eligible populations and splits as original.
No solver, no branch-and-bound is run.
"""
from __future__ import annotations

from datetime import datetime
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Tuple

import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import (
    GradientBoostingClassifier,
    GradientBoostingRegressor,
    RandomForestClassifier,
    RandomForestRegressor,
)
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

try:
    from xgboost import XGBClassifier, XGBRegressor
except Exception as e:
    raise ImportError(
        "xgboost is required because the final B&B experiment used XGB-C/XGB-R."
    ) from e


# ============================================================
# CONFIG — LOCKED TO FINAL B&B EXPERIMENT
# ============================================================

ROOT = Path(r"C:\Users\suuser\Desktop\minlplib")
FINAL_DIR = ROOT / "Areview" / "combined_minlplib_qplib"
DIM_TAG = "combined_minlplib_qplib"
SYNTHETIC_FINAL_DIR = (
    ROOT / "paperexperiments" / "synthetic_prime_minimal_revision"
    / "bb_separate_trees" / "synthetic_prime_8016_fixed100_v1"
    / "postrun_clean"
)

SKLEARN_RUN_NAME = "sklearn_all_runs_synthetic_prime_8016_20261021_014902_034048"


RANDOM_STATE = 42
TARGET_COL = "delta_root"
CLASS_COL = "y_root"

TAU = 0.01
ETA = 1e-3

USE_SAMPLE_WEIGHTS = False
WEIGHT_ALPHA = 0.30

EXPECTED_STATIC_FEATURES = 64
EXPECTED_COMBINED = 280
EXPECTED_SYNTHETIC = 6069
EXPECTED_ROOT_COMBINED = 278
EXPECTED_ROOT_SYNTHETIC = 6068
ROOT_ONLY_SYNTHETIC_EXCLUDE_KEYS = {"SYNTHETIC||5049"}
ROOT_ONLY_EXCLUDE_KEYS = {
    "MINLPLIB||ex5_3_3",
    "QPLIB||QPLIB_9048",
}
# These two exclusions apply ONLY to root-node model evaluation, not B&B.

EXCLUDE_COMBINED_KEYS = {
    "QPLIB||QPLIB_2512",
    "QPLIB||QPLIB_3402",
    "QPLIB||QPLIB_3703",
}

# Use the same final B&B target files ONLY to recover the exact eligible
# instance sets. Their B&B delta/y values are NOT used as root targets.
BB_TARGET_PROBE = 10

COMBINED_BB_TARGET = (
    FINAL_DIR
    / "targets"
    / "method2"
    / DIM_TAG
    / f"targets_method2_relaxation_{DIM_TAG}_probe{BB_TARGET_PROBE}.csv"
)

SYNTHETIC_BB_TARGET = (SYNTHETIC_FINAL_DIR / "targets" / "method2"
                       / "synthetic_prime" / f"targets_method2_relaxation_synthetic_prime_probe{BB_TARGET_PROBE}.csv")

# Same static feature files as final B&B static runs.
COMBINED_STATIC = (
    FINAL_DIR
    / "combined_minlplib_qplib_features_DI_64_clean.xlsx"
)

SYNTHETIC_STATIC = SYNTHETIC_FINAL_DIR / "synthetic_prime_DI_64_clean.xlsx"

# Root objectives stored in the clean summary files.
COMBINED_SUMMARY = (
    FINAL_DIR
    / "summary_combined_minlplib_qplib_BB_clean.csv"
)

SYNTHETIC_SUMMARY = SYNTHETIC_FINAL_DIR / "synthetic_prime_BB_summary.csv"

# SAME exact Combined split generated by the final B&B sklearn runner.
BB_OUTPUT_ROOT = FINAL_DIR / SKLEARN_RUN_NAME

COMBINED_SPLIT_FILE = (
    BB_OUTPUT_ROOT
    / "combined_train_test_split_manifest.csv"
)

RUN_STAMP = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
OUTPUT_ROOT = FINAL_DIR / f"sklearn_root_8_designs_278_two_excluded_{RUN_STAMP}"

MASTER_RESULTS_FILE = (
    OUTPUT_ROOT
    / "all_root_only_model_results.csv"
)

DESIGN_RESULTS_FILE = (
    OUTPUT_ROOT
    / "root_only_design_summary.csv"
)

EXPERIMENT_MANIFEST_FILE = (
    OUTPUT_ROOT
    / "experiment_manifest.csv"
)


# ============================================================
# HELPERS
# ============================================================

def _ts():
    return datetime.now().strftime("%H:%M:%S")


def read_any(path: Path) -> pd.DataFrame:
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f"Required file not found:\n{path}")

    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)

    if path.suffix.lower() in {".xlsx", ".xls"}:
        return pd.read_excel(path)

    raise ValueError(f"Unsupported file type: {path}")


def _canon_name(x) -> str:
    s = str(x).strip()
    if s.lower().endswith(".pkl"):
        s = s[:-4]
    return s


def _norm_source(x) -> str:
    return str(x).strip().upper()


def _require_columns(df: pd.DataFrame, cols: Iterable[str], label: str):
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ValueError(
            f"{label} missing columns {missing}. "
            f"Available columns: {list(df.columns)}"
        )


def _assert_unique(df: pd.DataFrame, key: str, label: str):
    if df[key].duplicated().any():
        ex = (
            df.loc[df[key].duplicated(keep=False), key]
            .astype(str)
            .head(20)
            .tolist()
        )
        raise ValueError(f"{label}: duplicate {key}. Examples: {ex}")


# ============================================================
# KEY HANDLING — SAME CONVENTIONS AS FINAL B&B SCRIPT
# ============================================================

def canonicalize_combined_key(df: pd.DataFrame, label: str) -> pd.DataFrame:
    out = df.copy()

    if "instance_key" in out.columns:
        out["_key"] = out["instance_key"].astype(str).str.strip()
    else:
        if "instance_name" not in out.columns:
            raise ValueError(
                f"{label}: need instance_key or instance_name."
            )

        out["instance_name"] = out["instance_name"].map(_canon_name)

        if "source" in out.columns:
            src = out["source"].map(_norm_source)

        elif "id" in out.columns:
            ids = out["id"].astype(str).str.strip()

            src = pd.Series(
                np.where(
                    ids.str.startswith("MINLPLIB__"),
                    "MINLPLIB",
                    np.where(
                        ids.str.startswith("QPLIB__"),
                        "QPLIB",
                        "",
                    ),
                ),
                index=out.index,
                dtype=object,
            )

            if (src == "").any():
                bad = out.loc[src == "", "id"].head(20).tolist()
                raise ValueError(
                    f"{label}: cannot infer source from id. Examples: {bad}"
                )
        else:
            raise ValueError(f"{label}: cannot construct Combined key.")

        out["_key"] = (
            src.astype(str)
            + "||"
            + out["instance_name"].astype(str)
        )

    out["_key"] = out["_key"].astype(str).str.strip()

    if "source" not in out.columns:
        out["source"] = out["_key"].str.split("||", n=1).str[0]

    out["source"] = out["source"].map(_norm_source)

    if "instance_name" not in out.columns:
        out["instance_name"] = out["_key"].str.split("||", n=1).str[1]

    out["instance_name"] = out["instance_name"].map(_canon_name)

    _assert_unique(out, "_key", label)
    return out


def canonicalize_synthetic_key(
    df: pd.DataFrame, label: str, target_keys: Optional[set] = None
) -> pd.DataFrame:
    """Canonical SYNTHETIC||<original-id>; never join by row positions."""
    out = df.copy()
    candidates = [c for c in ("instance_key", "dataset_id", "instance_id", "id")
                  if c in out.columns]
    if not candidates:
        raise ValueError(f"{label}: no original numeric ID or instance_key")
    best_col, best_keys, best_overlap = None, None, -1
    for col in candidates:
        raw = out[col].astype("string").str.strip()
        raw = raw.str.replace(r"^SYNTHETIC\|\|", "", regex=True)
        if raw.isna().any() or not raw.str.fullmatch(r"[0-9]+").all():
            continue
        keys = "SYNTHETIC||" + raw.astype(str)
        overlap = len(set(keys) & target_keys) if target_keys is not None else len(keys)
        if overlap > best_overlap:
            best_col, best_keys, best_overlap = col, keys, overlap
    if best_keys is None or (target_keys is not None and best_overlap == 0):
        raise ValueError(f"{label}: cannot align original Synthetic' IDs to targets")
    out["_key"] = best_keys
    _assert_unique(out, "_key", label)
    print(f"[{_ts()}] [{label}] matching on {best_col}; overlap={best_overlap}")
    return out


# ============================================================
# EXACT B&B ELIGIBLE INSTANCE SETS
# ============================================================

def load_bb_eligible_keys() -> Tuple[pd.DataFrame, pd.DataFrame]:
    combined = canonicalize_combined_key(
        read_any(COMBINED_BB_TARGET),
        "Combined final B&B target",
    )

    synthetic = canonicalize_synthetic_key(
        read_any(SYNTHETIC_BB_TARGET),
        "Synthetic' final B&B target",
    )

    # Same explicit three Combined exclusions as the final B&B experiment.
    combined = (
        combined.loc[
            ~combined["_key"].isin(EXCLUDE_COMBINED_KEYS)
        ]
        .copy()
        .reset_index(drop=True)
    )

    if len(combined) != EXPECTED_COMBINED:
        raise RuntimeError(
            f"Expected exactly {EXPECTED_COMBINED} final Combined B&B instances, "
            f"found {len(combined)}."
        )

    if len(synthetic) != EXPECTED_SYNTHETIC:
        raise RuntimeError(
            f"Expected exactly {EXPECTED_SYNTHETIC} final Synthetic' B&B instances, "
            f"found {len(synthetic)}."
        )

    _assert_unique(combined, "_key", "Combined B&B eligible set")
    _assert_unique(synthetic, "_key", "Synthetic' B&B eligible set")

    print(
        f"[{_ts()}] Locked B&B eligible sets: "
        f"Combined={len(combined)}, Synthetic'={len(synthetic)}",
        flush=True,
    )

    return combined, synthetic


# ============================================================
# ROOT TARGETS
# ============================================================

# ============================================================
# THREE ROOT OBJECTIVES — FROM ROOT SOLVE OUTPUT FILES, NOT B&B LABELS
# ============================================================
# Input tables may have extra instances. Eligible IDs stay locked to the original
# B&B experiment. We never determine eligibility from the objective tables.
# Set explicit paths below if automatic discovery cannot unambiguously find them.
# Authoritative ROOT SOLVE result tables: never scan unrelated archives or B&B targets.
QPLIB_ROOT_FILES = [ROOT / "Areview" / "qplib" / "qplib_root_results_ALL143.csv"]
MINLPLIB_ROOT_FILES = [ROOT / "minlplib_objectives_final.csv"]
SYNTHETIC_ROOT_FILES = [ROOT / "paperexperiments" / "synthetic_prime_minimal_revision" / "instances_mixed_8016_root_results.csv"]

LP_COLUMNS = ("lp_obj",)
SDP_COLUMNS = ("sdp_obj",)
SDPP_COLUMNS = ("sdpp_obj", "sdp_prime_obj")


def _status_and_objective(df, prefix, value_col, alias_status):
    values = pd.to_numeric(df[value_col], errors="coerce").to_numpy(dtype=float)
    status_col = next((c for c in alias_status if c in df.columns), None)
    if status_col is None:
        statuses = np.where(np.isneginf(values), "unbounded", np.where(np.isfinite(values), "optimal_unverified", "missing"))
    else:
        statuses = df[status_col].astype(str).str.strip().str.lower().to_numpy()
        # A stored -inf is a mathematical unbounded result only when status agrees.
        for i, (v, st) in enumerate(zip(values, statuses)):
            if np.isneginf(v) and st not in ("unbounded", "dual_infeasible", "primal_unbounded"):
                raise RuntimeError(f"{prefix}: -inf objective but status {st!r} at row {i}")
            if np.isfinite(v) and st not in ("optimal", "optimal_unverified"):
                values[i] = np.nan  # only trust solved finite objectives
            if np.isposinf(v):
                values[i] = np.nan
    return values, statuses


def _canonical_result_keys(df, origin):
    if origin == "minlplib":
        if "source_pkl" in df.columns:
            # e.g. 'ex3_1_3.pkl', unlike 'instance_name' = 'ex3_1_3_23'.
            names = df["source_pkl"].map(lambda v: Path(str(v)).stem)
        elif "id_global" in df.columns:
            names = df["id_global"].astype(str).str.split("__", regex=False).str[0]
        else:
            raise RuntimeError("MINLPLib objective CSV needs source_pkl or id_global for identity")
        return "MINLPLIB||" + names.astype(str)
    if origin == "qplib":
        if "name" not in df.columns:
            raise RuntimeError("QPLIB root CSV needs name")
        return "QPLIB||" + df["name"].astype(str).str.strip().map(_canon_name)
    if origin == "synthetic":
        if "id" not in df.columns:
            raise RuntimeError("Synthetic' root CSV needs original numeric id")
        ids = pd.to_numeric(df["id"], errors="coerce")
        if ids.isna().any() or (ids != np.floor(ids)).any():
            raise RuntimeError("Synthetic' root id must be integral")
        return "SYNTHETIC||" + ids.astype("int64").astype(str)
    raise ValueError(origin)


def _read_authoritative(path, origin, eligible):
    df = read_any(path).copy()
    df["_key"] = _canonical_result_keys(df, origin)
    df = df.loc[df["_key"].isin(eligible)].copy().reset_index(drop=True)
    if df["_key"].duplicated().any():
        raise RuntimeError(f"{path}: repeated eligible instance keys: {df.loc[df['_key'].duplicated(), '_key'].head(10).tolist()}")
    output = pd.DataFrame({"_key": df["_key"]})
    col_options = {"lp": LP_COLUMNS, "sdp": SDP_COLUMNS, "sdpp": SDPP_COLUMNS}
    for method, aliases in col_options.items():
        col = next((name for name in aliases if name in df.columns), None)
        if col is None:
            raise RuntimeError(f"{path}: missing {method} objective column; columns: {list(df.columns)}")
        values, statuses = _status_and_objective(
            df, f"{origin} {method}", col,
            [f"{method}_status", "sdp_prime_status" if method=="sdpp" else f"{method}_status"]
        )
        output[f"{method}_obj"] = values
        output[f"{method}_status"] = statuses
        output[f"{method}_source"] = str(path)
    return output


def load_three_root_objectives(kind, eligible):
    if kind == "synthetic":
        sources = [(SYNTHETIC_ROOT_FILES[0], "synthetic")]
    else:
        sources = [(MINLPLIB_ROOT_FILES[0], "minlplib"), (QPLIB_ROOT_FILES[0], "qplib")]
    frames = [_read_authoritative(path, origin, eligible) for path, origin in sources]
    assembled = pd.concat(frames, ignore_index=True)
    _assert_unique(assembled, "_key", f"{kind} authoritative root sources")
    complete_keys = set(assembled["_key"])
    absent = eligible - complete_keys
    if absent:
        audit = OUTPUT_ROOT / f"MISSING_{kind.upper()}_ROOT_OBJECTIVES.csv"
        pd.DataFrame({"_key": sorted(absent)}).to_csv(audit, index=False)
        raise RuntimeError(f"{kind}: {len(absent)} locked instances have no objective record; audit {audit}. First: {sorted(absent)[:15]}")
    assembled = assembled.set_index("_key").loc[sorted(eligible)].reset_index()
    report = OUTPUT_ROOT / f"root_objective_status_audit_{kind}.csv"
    assembled.to_csv(report, index=False)
    for method in ("lp", "sdp", "sdpp"):
        vals = assembled[f"{method}_obj"].to_numpy(float)
        print(f"[{_ts()}] {kind} {method}: finite={np.isfinite(vals).sum()}, unbounded_below={np.isneginf(vals).sum()}, other_missing={np.isnan(vals).sum()}", flush=True)
    # LP and SDP-prime need finite objectives for the two experiments.
    # An SDP=-inf value is valid and receives delta=+1 by a declared convention.
    for method in ("lp", "sdpp"):
        invalid = ~np.isfinite(assembled[f"{method}_obj"].to_numpy(float))
        if invalid.any():
            bad = assembled.loc[invalid, ["_key", f"{method}_obj", f"{method}_status"]]
            raise RuntimeError(f"{kind}: {len(bad)} locked instances do not have finite {method} objectives. No rows removed. Audit {report}.\n{bad.head(20).to_string(index=False)}")
    sdp = assembled["sdp_obj"].to_numpy(float)
    bad = ~(np.isfinite(sdp) | np.isneginf(sdp))
    if bad.any():
        rows = assembled.loc[bad, ["_key", "sdp_obj", "sdp_status"]]
        raise RuntimeError(f"{kind}: {len(rows)} SDP outcomes neither finite nor unbounded below. No rows removed. Audit {report}.\n{rows.head(20).to_string(index=False)}")
    return assembled

def compute_root_targets(root_frame, dataset_kind, eligible_keys, comparison):
    if set(root_frame['_key']) != eligible_keys or len(root_frame)!=len(eligible_keys):
        raise RuntimeError(f"{dataset_kind}: root objectives do not exactly match locked eligible IDs")
    out=root_frame.copy()
    method='sdp' if comparison=='LP_vs_SDP' else 'sdpp'
    zlp=out['lp_obj'].to_numpy(float)
    zother=out[f'{method}_obj'].to_numpy(float)
    if not np.isfinite(zlp).all():
        raise RuntimeError(f"{dataset_kind}: nonfinite LP objective")
    if comparison=='LP_vs_SDP_PRIME' and not np.isfinite(zother).all():
        raise RuntimeError(f"{dataset_kind}: nonfinite SDP-prime objective")
    if comparison=='LP_vs_SDP' and not (np.isfinite(zother) | np.isneginf(zother)).all():
        raise RuntimeError(f"{dataset_kind}: invalid SDP statuses: check root audit")
    out['lp_root_obj']=zlp
    out['sdp_root_obj']=zother # compatibility with original model runner
    # Mathematical limit of (LP - SDP)/(|LP|+|SDP|+ETA) as SDP -> -infinity is +1.
    # Explicitly record the unbounded cases; do not treat them as missing or drop.
    unbounded=np.isneginf(zother)
    delta=np.empty(len(zlp), dtype=float)
    delta[unbounded]=1.0
    finite=~unbounded
    delta[finite]=(zlp[finite]-zother[finite])/(np.abs(zlp[finite])+np.abs(zother[finite])+ETA)
    out['sdp_unbounded_below']=unbounded if comparison=='LP_vs_SDP' else False
    out[TARGET_COL]=delta
    out[CLASS_COL]=(out[TARGET_COL].to_numpy(float)>-TAU).astype(int)
    out['source']= 'SYNTHETIC' if dataset_kind=='synthetic' else out['_key'].str.split('||',regex=False).str[0]
    out['instance_name']=out['_key'].str.split('||',regex=False).str[-1]
    out['root_winner_thresholded']=np.where(out[CLASS_COL]==1,'LP',method.upper())
    out['comparison']=comparison
    print(f"[{_ts()}] {dataset_kind} {comparison}: labels {out[CLASS_COL].value_counts().sort_index().to_dict()}")
    return out


# ============================================================
# STATIC FEATURE LOADING — EXACT SAME 64 FEATURES
# ============================================================

FORBIDDEN_STATIC_COLS = {
    "id",
    "id_key",
    "id_global",
    "instance_id",
    "instanceid",
    "instance_index",
    "instance_name",
    "instance_key",
    "problem_name",
    "name",
    "source",
    "_key",
    "y",
    "winner",
    "delta",
    "y_root",
    "delta_root",
    "lp_obj",
    "sdp_prime_obj",
    "sdp_obj",
    "sdpp_obj",
    "lp_root_obj",
    "sdp_root_obj",
    "lp_status",
    "sdp_status",
    "sdp_prime_status",
    "sdpp_status",
}


def load_static_features(
    dataset_kind: str,
    eligible_keys: set,
) -> Tuple[pd.DataFrame, List[str]]:
    path = (
        COMBINED_STATIC
        if dataset_kind == "combined"
        else SYNTHETIC_STATIC
    )

    raw = read_any(path)

    if dataset_kind == "combined":
        df = canonicalize_combined_key(
            raw,
            "Combined static features",
        )
    else:
        df = canonicalize_synthetic_key(
            raw,
            "Synthetic' static features",
            target_keys=eligible_keys,
        )

    feature_cols = [
        c
        for c in df.columns
        if c not in FORBIDDEN_STATIC_COLS
    ]

    if len(feature_cols) != EXPECTED_STATIC_FEATURES:
        raise RuntimeError(
            f"{dataset_kind}: expected exactly {EXPECTED_STATIC_FEATURES} "
            f"static DI features, found {len(feature_cols)}.\n"
            f"Columns: {feature_cols}"
        )

    missing = eligible_keys - set(df["_key"])
    if missing:
        raise RuntimeError(
            f"{dataset_kind}: static feature file misses B&B-eligible instances. "
            f"Examples: {sorted(missing)[:30]}"
        )

    out = (
        df.loc[df["_key"].isin(eligible_keys), ["_key"] + feature_cols]
        .copy()
        .reset_index(drop=True)
    )

    for c in feature_cols:
        out[c] = pd.to_numeric(out[c], errors="coerce")

    _assert_unique(out, "_key", f"{dataset_kind} static features")

    return out, feature_cols


# ============================================================
# SAME B&B COMBINED SPLIT
# ============================================================

def load_same_combined_split(
    combined_keys: set,
) -> Tuple[set, set, pd.DataFrame]:
    m = read_any(COMBINED_SPLIT_FILE)

    _require_columns(
        m,
        ["_key", "split"],
        "final B&B Combined split manifest",
    )

    m["_key"] = m["_key"].astype(str).str.strip()
    m["split"] = m["split"].astype(str).str.strip().str.lower()

    _assert_unique(m, "_key", "final B&B Combined split manifest")

    if set(m["_key"]) != combined_keys:
        raise RuntimeError(
            "The B&B split manifest does not match the final 280-instance "
            "Combined set."
        )

    if set(m["split"]) - {"train", "test"}:
        raise RuntimeError(
            "B&B split manifest contains values other than train/test."
        )

    train_keys = set(m.loc[m["split"] == "train", "_key"])
    test_keys = set(m.loc[m["split"] == "test", "_key"])

    if train_keys & test_keys:
        raise RuntimeError("B&B split manifest has train/test overlap.")

    print(
        f"[{_ts()}] Reusing exact B&B Combined split: "
        f"train={len(train_keys)}, test={len(test_keys)}",
        flush=True,
    )

    return train_keys, test_keys, m


# ============================================================
# MODELS — EXACT SAME HYPERPARAMETERS AS FINAL B&B RUNNER
# ============================================================

def make_models(task: str) -> Dict[str, object]:
    if task == "classification":
        return {
            "XGB-C": XGBClassifier(
                n_estimators=400,
                learning_rate=0.08,
                max_depth=6,
                subsample=0.8,
                colsample_bytree=0.8,
                reg_alpha=0.1,
                reg_lambda=1.0,
                random_state=RANDOM_STATE,
                n_jobs=-1,
                eval_metric="logloss",
            ),
            "GBC": GradientBoostingClassifier(
                random_state=RANDOM_STATE,
            ),
            "RFC": RandomForestClassifier(
                n_estimators=400,
                max_features="sqrt",
                class_weight="balanced",
                n_jobs=-1,
                random_state=RANDOM_STATE,
            ),
        }

    if task == "regression":
        return {
            "XGB-R": XGBRegressor(
                n_estimators=400,
                learning_rate=0.08,
                max_depth=6,
                subsample=0.8,
                colsample_bytree=0.8,
                reg_alpha=0.1,
                reg_lambda=1.0,
                random_state=RANDOM_STATE,
                n_jobs=-1,
            ),
            "GBR": GradientBoostingRegressor(
                random_state=RANDOM_STATE,
            ),
            "RFR": RandomForestRegressor(
                n_estimators=400,
                n_jobs=-1,
                random_state=RANDOM_STATE,
            ),
        }

    raise ValueError(f"Unknown task={task}")


def build_pipeline(model) -> Pipeline:
    try:
        imputer = SimpleImputer(
            strategy="median",
            keep_empty_features=True,
        )
    except TypeError:
        imputer = SimpleImputer(
            strategy="median",
        )

    return Pipeline(
        [
            ("imputer", imputer),
            ("model", model),
        ]
    )


# ============================================================
# METRICS — SAME AS FINAL B&B RUNNER
# ============================================================

def safe_pearson_r(y_true, y_pred):
    yt = np.asarray(y_true, dtype=float)
    yp = np.asarray(y_pred, dtype=float)

    mask = np.isfinite(yt) & np.isfinite(yp)
    yt = yt[mask]
    yp = yp[mask]

    if len(yt) < 2 or np.std(yt) <= 0 or np.std(yp) <= 0:
        return float("nan")

    r = np.corrcoef(yt, yp)[0, 1]
    return float(r) if np.isfinite(r) else float("nan")


def r_accuracy_paper(y_true_delta, y_pred_delta, eps=TAU):
    yt = np.asarray(y_true_delta, dtype=float)
    yp = np.asarray(y_pred_delta, dtype=float)

    mask = np.isfinite(yt) & np.isfinite(yp)
    yt = yt[mask]
    yp = yp[mask]

    if len(yt) == 0:
        return float("nan")

    same_sign = np.sign(yt) == np.sign(yp)

    forgiveness = (
        (yt > -eps)
        & (yt < 0)
        & (yp > 0)
    )

    return float(np.mean(same_sign | forgiveness))


def r_accuracy_tol(y_true_delta, y_pred_delta, eps=TAU):
    yt = np.asarray(y_true_delta, dtype=float)
    yp = np.asarray(y_pred_delta, dtype=float)

    mask = np.isfinite(yt) & np.isfinite(yp)
    yt = yt[mask]
    yp = yp[mask]

    if len(yt) == 0:
        return float("nan")

    same = np.sign(yt) == np.sign(yp)

    q2 = (
        (yt > -eps)
        & (yt < 0)
        & (yp > 0)
    )

    q4 = (
        (yt > 0)
        & (yt < eps)
        & (yp < 0)
        & (yp > -eps)
    )

    return float(np.mean(same | q2 | q4))


def decision_from_delta(delta_values):
    return (
        np.asarray(delta_values, dtype=float)
        > -TAU
    ).astype(int)


# ============================================================
# MERGE
# ============================================================

def merge_targets_features(
    target_df: pd.DataFrame,
    feature_df: pd.DataFrame,
    feature_cols: List[str],
    label: str,
) -> pd.DataFrame:
    missing = set(target_df["_key"]) - set(feature_df["_key"])

    if missing:
        raise RuntimeError(
            f"{label}: targets missing static features. "
            f"Examples: {sorted(missing)[:30]}"
        )

    target_cols = [
        c
        for c in [
            "_key",
            "source",
            "instance_index",
            "instance_id",
            "instance_name",
            "lp_root_obj",
            "sdp_root_obj",
            TARGET_COL,
            CLASS_COL,
        ]
        if c in target_df.columns
    ]

    out = target_df[target_cols].merge(
        feature_df[["_key"] + feature_cols],
        on="_key",
        how="left",
        validate="one_to_one",
    )

    if len(out) != len(target_df):
        raise RuntimeError(f"{label}: merge changed row count.")

    return out


# ============================================================
# ONE DESIGN
# ============================================================

def run_design(
    *,
    design_id: int,
    train_setting: str,
    task: str,
    combined_root: pd.DataFrame,
    comparison: str,
    synthetic_root: pd.DataFrame,
    combined_features: pd.DataFrame,
    synthetic_features: pd.DataFrame,
    feature_cols: List[str],
    combined_train_keys: set,
    combined_test_keys: set,
) -> List[dict]:
    if train_setting == "combined_to_combined":
        all_combined = merge_targets_features(
            combined_root,
            combined_features,
            feature_cols,
            "Combined root",
        )

        train_df = (
            all_combined.loc[
                all_combined["_key"].isin(combined_train_keys)
            ]
            .copy()
            .reset_index(drop=True)
        )

        test_df = (
            all_combined.loc[
                all_combined["_key"].isin(combined_test_keys)
            ]
            .copy()
            .reset_index(drop=True)
        )

        setting_label = "Combined→Combined"

    elif train_setting == "synthetic_to_combined":
        # IMPORTANT: exactly mirrors the final B&B sklearn runner.
        # Train on ALL Synthetic'; test on ALL 280 Combined.
        train_df = merge_targets_features(
            synthetic_root,
            synthetic_features,
            feature_cols,
            "Synthetic' root train",
        )

        test_df = merge_targets_features(
            combined_root,
            combined_features,
            feature_cols,
            "Combined root transfer test",
        )

        setting_label = "Synthetic′→Combined"

    else:
        raise ValueError(train_setting)

    if task == "classification":
        y_train = train_df[CLASS_COL].to_numpy(int)
        y_test = test_df[CLASS_COL].to_numpy(int)

        if len(np.unique(y_train)) < 2:
            raise RuntimeError(
                f"Design {design_id}: classification train set is single-class."
            )
    else:
        y_train = None
        y_test = test_df[CLASS_COL].to_numpy(int)

    d_train = train_df[TARGET_COL].to_numpy(float)
    d_test = test_df[TARGET_COL].to_numpy(float)

    X_train = train_df[feature_cols].replace([np.inf, -np.inf], np.nan)
    X_test = test_df[feature_cols].replace([np.inf, -np.inf], np.nan)

    run_dir = (
        OUTPUT_ROOT
        / f"design_{design_id:02d}__{comparison}__{train_setting}__{task}"
    )

    run_dir.mkdir(parents=True, exist_ok=True)

    split_out = pd.concat(
        [
            train_df[["_key", TARGET_COL, CLASS_COL]].assign(split="train"),
            test_df[["_key", TARGET_COL, CLASS_COL]].assign(split="test"),
        ],
        ignore_index=True,
    )

    split_out.to_csv(
        run_dir / "instance_split_root_targets.csv",
        index=False,
    )

    pd.DataFrame({"feature": feature_cols}).to_csv(
        run_dir / "feature_columns.csv",
        index=False,
    )

    print("\n" + "=" * 88)
    print(
        f"[{_ts()}] DESIGN {design_id} | {setting_label} | ROOT {task.upper()}"
    )
    print(
        f"[{_ts()}] train={len(train_df)} test={len(test_df)} "
        f"features={len(feature_cols)}"
    )
    print(
        f"[{_ts()}] root train labels="
        f"{train_df[CLASS_COL].value_counts().sort_index().to_dict()} | "
        f"test labels="
        f"{test_df[CLASS_COL].value_counts().sort_index().to_dict()}"
    )
    print("=" * 88)

    result_rows = []

    for model_name, model in make_models(task).items():
        print(f"[{_ts()}] training {model_name}...", flush=True)

        pipe = build_pipeline(model)

        if task == "classification":
            pipe.fit(X_train, y_train)

            pred = pipe.predict(X_test).astype(int)

            if hasattr(pipe, "predict_proba") and len(np.unique(y_test)) > 1:
                prob = pipe.predict_proba(X_test)[:, 1]
                auc = float(roc_auc_score(y_test, prob))
            else:
                prob = np.full(len(y_test), np.nan)
                auc = float("nan")

            acc = float(accuracy_score(y_test, pred))
            bal = float(balanced_accuracy_score(y_test, pred))
            f1_lp = float(f1_score(y_test, pred, zero_division=0))
            f1_macro = float(
                f1_score(
                    y_test,
                    pred,
                    average="macro",
                    zero_division=0,
                )
            )
            precision = float(
                precision_score(y_test, pred, zero_division=0)
            )
            recall = float(
                recall_score(y_test, pred, zero_division=0)
            )

            cm = confusion_matrix(y_test, pred, labels=[0, 1])

            pd.DataFrame(
                cm,
                index=[f"true_{comparison.split('_vs_')[-1]}(0)", "true_LP(1)"],
                columns=[f"pred_{comparison.split('_vs_')[-1]}(0)", "pred_LP(1)"],
            ).to_csv(
                run_dir / f"cm_{model_name}.csv",
                index=True,
            )

            disp = ConfusionMatrixDisplay(
                cm,
                display_labels=[comparison.split("_vs_")[-1], "LP"],
            )
            disp.plot()
            plt.title(
                f"Root | {setting_label} | {model_name}"
            )
            plt.tight_layout()
            plt.savefig(
                run_dir / f"cm_{model_name}.png",
                dpi=200,
            )
            plt.close()

            pred_df = test_df[
                ["_key", TARGET_COL, CLASS_COL]
            ].copy()

            pred_df["y_pred"] = pred
            pred_df["prob_LP"] = prob

            pred_df.to_csv(
                run_dir / f"preds_{model_name}.csv",
                index=False,
            )

            result = {
                "design_id": design_id,
                "comparison": comparison,
                "train_test": setting_label,
                "task": task,
                "input": "64_static_DI",
                "target": "root_" + comparison,
                "model": model_name,
                "n_train": len(train_df),
                "n_test": len(test_df),
                "accuracy": acc,
                "balanced_accuracy": bal,
                "f1_lp": f1_lp,
                "f1_macro": f1_macro,
                "precision_LP": precision,
                "recall_LP": recall,
                "roc_auc": auc,
                "rmse": np.nan,
                "mae": np.nan,
                "r2": np.nan,
                "pearson_r": np.nan,
                "r_accuracy_paper": np.nan,
                "r_accuracy_tol": np.nan,
                "decision_accuracy": np.nan,
            }

            print(
                f"[{_ts()}] {model_name}: "
                f"acc={acc:.4f} bal={bal:.4f} macroF1={f1_macro:.4f}",
                flush=True,
            )

        else:
            pipe.fit(X_train, d_train)

            delta_pred = pipe.predict(X_test)
            pred_class = decision_from_delta(delta_pred)

            mse = float(
                mean_squared_error(
                    d_test,
                    delta_pred,
                )
            )
            rmse = float(np.sqrt(mse))
            mae = float(
                mean_absolute_error(
                    d_test,
                    delta_pred,
                )
            )
            r2 = float(
                r2_score(
                    d_test,
                    delta_pred,
                )
            )
            pearson = safe_pearson_r(
                d_test,
                delta_pred,
            )
            r_paper = r_accuracy_paper(
                d_test,
                delta_pred,
            )
            r_tol = r_accuracy_tol(
                d_test,
                delta_pred,
            )
            decision_acc = float(
                accuracy_score(
                    y_test,
                    pred_class,
                )
            )

            pred_df = test_df[
                ["_key", TARGET_COL, CLASS_COL]
            ].copy()

            pred_df["delta_pred"] = delta_pred
            pred_df["y_pred_from_delta"] = pred_class

            pred_df.to_csv(
                run_dir / f"preds_{model_name}.csv",
                index=False,
            )

            result = {
                "design_id": design_id,
                "comparison": comparison,
                "train_test": setting_label,
                "task": task,
                "input": "64_static_DI",
                "target": "root_" + comparison,
                "model": model_name,
                "n_train": len(train_df),
                "n_test": len(test_df),
                "accuracy": np.nan,
                "balanced_accuracy": np.nan,
                "f1_lp": np.nan,
                "f1_macro": np.nan,
                "precision_LP": np.nan,
                "recall_LP": np.nan,
                "roc_auc": np.nan,
                "rmse": rmse,
                "mae": mae,
                "r2": r2,
                "pearson_r": pearson,
                "r_accuracy_paper": r_paper,
                "r_accuracy_tol": r_tol,
                "decision_accuracy": decision_acc,
            }

            print(
                f"[{_ts()}] {model_name}: "
                f"rAcc_paper={r_paper:.4f} "
                f"rAcc_tol={r_tol:.4f} "
                f"RMSE={rmse:.5f} MAE={mae:.5f} R2={r2:.4f}",
                flush=True,
            )

        joblib.dump(
            pipe,
            run_dir / f"model_{model_name}.joblib",
        )

        result_rows.append(result)

    pd.DataFrame(result_rows).to_csv(
        run_dir / "results_all_models.csv",
        index=False,
    )

    return result_rows


# ============================================================
# MAIN
# ============================================================

def main():
    required=[COMBINED_BB_TARGET,SYNTHETIC_BB_TARGET,COMBINED_STATIC,
              SYNTHETIC_STATIC,COMBINED_SPLIT_FILE]
    missing=[str(p) for p in required if not Path(p).exists()]
    if missing:
        raise FileNotFoundError("Missing locked experiment inputs:\n"+"\n".join(missing))
    if OUTPUT_ROOT.exists():
        raise FileExistsError(f"Refusing to overwrite: {OUTPUT_ROOT}")
    OUTPUT_ROOT.mkdir(parents=True,exist_ok=False)
    print("ROOT NODE ONLY: 8 DESIGNS / 24 MODELS; NO B&B SOLVING",flush=True)
    bb_combined,bb_synthetic=load_bb_eligible_keys()
    combined_keys=set(bb_combined['_key'])
    synthetic_keys=set(bb_synthetic['_key'])
    combined_feat,combined_cols=load_static_features('combined',combined_keys)
    synthetic_feat,synthetic_cols=load_static_features('synthetic',synthetic_keys)
    if combined_cols!=synthetic_cols:
        raise RuntimeError('Different 64-feature schemas between Combined and Synthetic')
    combined_train_keys,combined_test_keys,split_manifest=load_same_combined_split(combined_keys)
    if len(combined_train_keys)!=177 or len(combined_test_keys)!=103:
        raise RuntimeError('Train/test counts changed from original 177/103')
    # IMPORTANT: validate the original 280-instance population and split FIRST.
    # Then remove only the two agreed root-only exclusions, keeping all remaining
    # original train/test assignments and leaving the B&B experiment unchanged.
    if not ROOT_ONLY_EXCLUDE_KEYS.issubset(combined_keys):
        raise RuntimeError(
            f"Root-only exclusions missing from original 280: "
            f"{sorted(ROOT_ONLY_EXCLUDE_KEYS - combined_keys)}"
        )
    excluded = split_manifest.loc[
        split_manifest['_key'].isin(ROOT_ONLY_EXCLUDE_KEYS), ['_key', 'split']
    ].copy()
    if set(excluded['_key']) != ROOT_ONLY_EXCLUDE_KEYS or len(excluded) != 2:
        raise RuntimeError('Could not audit both root-only exclusions against original split')
    excluded['reason'] = 'Full SDP objective unavailable (NaN; status unknown)'
    excluded.to_csv(OUTPUT_ROOT/'root_only_exclusions.csv', index=False)
    split_manifest.to_csv(
        OUTPUT_ROOT/'original_bb_combined_split_manifest_280.csv',index=False
    )

    combined_keys = combined_keys - ROOT_ONLY_EXCLUDE_KEYS
    combined_train_keys = combined_train_keys - ROOT_ONLY_EXCLUDE_KEYS
    combined_test_keys = combined_test_keys - ROOT_ONLY_EXCLUDE_KEYS
    split_manifest = split_manifest.loc[
        split_manifest['_key'].isin(combined_keys)
    ].copy().reset_index(drop=True)
    if (len(combined_keys) != EXPECTED_ROOT_COMBINED
            or set(split_manifest['_key']) != combined_keys
            or combined_train_keys & combined_test_keys
            or (combined_train_keys | combined_test_keys) != combined_keys):
        raise RuntimeError('Root-only 278-instance population/split validation failed')
    split_manifest.to_csv(OUTPUT_ROOT/'reused_bb_combined_split_manifest.csv',index=False)
    print(f"ROOT-ONLY exclusions: {sorted(ROOT_ONLY_EXCLUDE_KEYS)}",flush=True)
    print(f"ROOT-ONLY Combined: {len(combined_keys)} "
          f"(train={len(combined_train_keys)}, test={len(combined_test_keys)}); "
          f"Synthetic' remains {len(synthetic_keys)}",flush=True)

    # Validate original 6,069 Synthetic' IDs before applying the single agreed
    # root-only exclusion. Preserve the exact identity of all remaining cases.
    if not ROOT_ONLY_SYNTHETIC_EXCLUDE_KEYS.issubset(synthetic_keys):
        raise RuntimeError(
            f"Synthetic root-only exclusions are not in original 6069 set: "
            f"{sorted(ROOT_ONLY_SYNTHETIC_EXCLUDE_KEYS - synthetic_keys)}"
        )
    pd.DataFrame({"_key": sorted(synthetic_keys)}).to_csv(
        OUTPUT_ROOT / "original_synthetic_eligible_keys_6069.csv", index=False
    )
    synthetic_excluded = pd.DataFrame({
        "_key": sorted(ROOT_ONLY_SYNTHETIC_EXCLUDE_KEYS),
        "reason": ["Full SDP objective unavailable (NaN; status unknown)"],
        "applies_to": ["All eight root-only designs (Synthetic training only)"]
    })
    synthetic_excluded.to_csv(
        OUTPUT_ROOT / "root_only_synthetic_exclusions.csv", index=False
    )
    synthetic_keys = synthetic_keys - ROOT_ONLY_SYNTHETIC_EXCLUDE_KEYS
    if len(synthetic_keys) != EXPECTED_ROOT_SYNTHETIC:
        raise RuntimeError(f"Expected 6068 retained Synthetic instances, got {len(synthetic_keys)}")
    pd.DataFrame({"_key": sorted(synthetic_keys)}).to_csv(
        OUTPUT_ROOT / "root_only_synthetic_eligible_keys_6068.csv", index=False
    )
    print(f"ROOT-ONLY Synthetic': {len(synthetic_keys)} instances "
          f"(removed {sorted(ROOT_ONLY_SYNTHETIC_EXCLUDE_KEYS)})", flush=True)

    print('Loading PRECOMPUTED root objective files only (no QCQP solving)',flush=True)
    combined_objectives=load_three_root_objectives('combined',combined_keys)
    synthetic_objectives=load_three_root_objectives('synthetic',synthetic_keys)
    combined_objectives.to_csv(OUTPUT_ROOT/'root_objective_sources_combined.csv',index=False)
    synthetic_objectives.to_csv(OUTPUT_ROOT/'root_objective_sources_synthetic.csv',index=False)

    plan=[]
    for comp in ('LP_vs_SDP','LP_vs_SDP_PRIME'):
        for setting,task in (("combined_to_combined","classification"),
                             ("combined_to_combined","regression"),
                             ("synthetic_to_combined","classification"),
                             ("synthetic_to_combined","regression")):
            plan.append((len(plan)+1,comp,setting,task))
    pd.DataFrame(plan,columns=['design_id','comparison','train_setting','task']).to_csv(
        EXPERIMENT_MANIFEST_FILE,index=False)
    all_results=[]
    for did,comp,setting,task in plan:
        combined_root=compute_root_targets(combined_objectives,'combined',combined_keys,comp)
        synthetic_root=compute_root_targets(synthetic_objectives,'synthetic',synthetic_keys,comp)
        combined_root.to_csv(OUTPUT_ROOT/f'root_targets_combined_{comp}.csv',index=False)
        synthetic_root.to_csv(OUTPUT_ROOT/f'root_targets_synthetic_{comp}.csv',index=False)
        rows=run_design(design_id=did,comparison=comp,train_setting=setting,task=task,
                        combined_root=combined_root,synthetic_root=synthetic_root,
                        combined_features=combined_feat,synthetic_features=synthetic_feat,
                        feature_cols=combined_cols,combined_train_keys=combined_train_keys,
                        combined_test_keys=combined_test_keys)
        all_results.extend(rows)
        pd.DataFrame(all_results).to_csv(MASTER_RESULTS_FILE,index=False)
    df=pd.DataFrame(all_results)
    if len(df)!=24:
        raise RuntimeError(f'Expected 24 model results; got {len(df)}')
    summary=[]
    for did,group in df.groupby('design_id',sort=True):
        metric='accuracy' if group['task'].iloc[0]=='classification' else 'r_accuracy_paper'
        best=group.loc[group[metric].astype(float).idxmax()]
        summary.append(dict(design_id=did,comparison=best['comparison'],
                            train_test=best['train_test'],task=best['task'],
                            best_model=best['model'],primary_metric=metric,primary_value=best[metric]))
    pd.DataFrame(summary).to_csv(DESIGN_RESULTS_FILE,index=False)
    print(f"COMPLETE: 8 designs, 24 models. Results -> {MASTER_RESULTS_FILE}",flush=True)

if __name__=='__main__':
    main()


ROOT NODE ONLY: 8 DESIGNS / 24 MODELS; NO B&B SOLVING
[20:10:10] [Synthetic' final B&B target] matching on instance_key; overlap=6069
[20:10:10] Locked B&B eligible sets: Combined=280, Synthetic'=6069
[20:10:13] [Synthetic' static features] matching on id; overlap=6069
[20:10:13] Reusing exact B&B Combined split: train=177, test=103
ROOT-ONLY exclusions: ['MINLPLIB||ex5_3_3', 'QPLIB||QPLIB_9048']
ROOT-ONLY Combined: 278 (train=177, test=101); Synthetic' remains 6069
ROOT-ONLY Synthetic': 6068 instances (removed ['SYNTHETIC||5049'])
Loading PRECOMPUTED root objective files only (no QCQP solving)
[20:10:13] combined lp: finite=278, unbounded_below=0, other_missing=0
[20:10:13] combined sdp: finite=209, unbounded_below=69, other_missing=0
[20:10:13] combined sdpp: finite=278, unbounded_below=0, other_missing=0
[20:10:14] synthetic lp: finite=6068, unbounded_below=0, other_missing=0
[20:10:14] synthetic sdp: finite=4769, unbounded_below=1299, other_missing=0
[20:10:14] synthetic sdpp: fini

In [11]:
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from sklearn.metrics import balanced_accuracy_score, mean_absolute_error

# Root-only feature importance for all 8 final root-node designs.
# Run after reviewrootnode(2).ipynb. Uses saved models and held-out predictions.
BASE = Path(r'C:\Users\suuser\Desktop\minlplib')
FINAL = BASE / 'Areview' / 'combined_minlplib_qplib'
ROOT_RUN = None   # Set an exact path to override auto-selection
FEATURES_FILE = FINAL / 'combined_minlplib_qplib_features_DI_64_clean.xlsx'
REPEATS = 20
SEED = 42
TOP_K = 15
OUT = FINAL / ('root_node_feature_importance_' + datetime.now().strftime('%Y%m%d_%H%M%S_%f'))

META = {'id','instance_id','instance_index','instance_name','instance_key','source','_key'}

def get_root_run():
    if ROOT_RUN is not None:
        candidate = Path(ROOT_RUN)
        if not (candidate / 'experiment_manifest.csv').is_file():
            raise FileNotFoundError(f'Missing experiment_manifest.csv: {candidate}')
        return candidate
    choices = sorted(FINAL.glob('sklearn_root_8_designs_278_two_excluded_*'), reverse=True)
    choices = [p for p in choices if (p / 'experiment_manifest.csv').is_file() and
               (p / 'all_root_only_model_results.csv').is_file()]
    if not choices:
        raise FileNotFoundError('Cannot find final 8-design root run; specify ROOT_RUN explicitly.')
    print('Selected most recent completed root-run:', choices[0])
    return choices[0]

def add_key(frame):
    f = frame.copy()
    if 'instance_key' in f:
        keys = f['instance_key'].astype(str).str.strip()
    elif 'instance_name' in f:
        names = f['instance_name'].astype(str).str.strip().str.replace(r'\.pkl$', '', regex=True)
        if 'source' in f:
            source = f['source'].astype(str).str.strip().str.upper()
        elif 'id' in f:
            source = pd.Series(np.where(f['id'].astype(str).str.startswith('QPLIB__'), 'QPLIB', 'MINLPLIB'), index=f.index)
        else:
            raise RuntimeError('Static DI data missing source information')
        keys = source + '||' + names
    elif 'id' in f:
        keys = f['id'].astype(str).str.strip()
    else:
        raise RuntimeError('Cannot derive ID from DI feature file')
    keys = keys.str.replace(r'\.0$', '', regex=True)
    f['_key'] = keys.map(lambda v: v if '||' in v else 'SYNTHETIC||' + v)
    if f['_key'].isna().any() or f['_key'].duplicated().any():
        raise RuntimeError('Missing or duplicate keys in static feature file')
    return f

def pick_model(results, design_id, task):
    rows = results[(results['design_id'].astype(int) == int(design_id)) &
                   (results['task'].astype(str).str.lower() == task)].copy()
    if len(rows) != 3:
        raise RuntimeError(f'Expected 3 fitted models for design {design_id}, found {len(rows)}')
    metric = 'accuracy' if task == 'classification' else 'r_accuracy_paper'
    rows[metric] = pd.to_numeric(rows[metric], errors='raise')
    row = rows.sort_values([metric, 'model'], ascending=[False, True]).iloc[0]
    return str(row['model']), metric, float(row[metric])

def one_design(root, static, entry, results, rng):
    did = int(entry['design_id'])
    comp = str(entry['comparison'])
    setting = str(entry['train_setting'])
    task = str(entry['task']).lower()
    model_name, select_metric, select_value = pick_model(results, did, task)
    folder = root / f'design_{did:02d}__{comp}__{setting}__{task}'
    model_file = folder / f'model_{model_name}.joblib'
    pred_file = folder / f'preds_{model_name}.csv'
    cols_file = folder / 'feature_columns.csv'
    for p in (model_file, pred_file, cols_file):
        if not p.is_file(): raise FileNotFoundError(p)
    columns = pd.read_csv(cols_file)['feature'].astype(str).tolist()
    if len(columns) != 64 or len(set(columns)) != 64:
        raise RuntimeError(f'Design {did}: unexpected DI feature schema length')
    absent = set(columns) - set(static.columns)
    if absent: raise RuntimeError(f'Design {did}: missing feature columns {sorted(absent)}')
    pred = pd.read_csv(pred_file)
    if '_key' not in pred: raise RuntimeError(f'Design {did}: predictions missing _key')
    pred['_key'] = pred['_key'].astype(str).str.strip()
    if pred['_key'].duplicated().any(): raise RuntimeError(f'Design {did}: duplicate prediction keys')
    df = pred.merge(static[['_key']+columns], how='left', on='_key', validate='one_to_one', indicator=True)
    if not df['_merge'].eq('both').all():
        raise RuntimeError(f'Design {did}: unmatched keys: {df.loc[df._merge.ne("both"),"_key"].head().tolist()}')
    X = df[columns].apply(pd.to_numeric, errors='coerce').replace([np.inf,-np.inf],np.nan)
    model = joblib.load(model_file)
    fresh = np.asarray(model.predict(X))
    if task == 'classification':
        if 'y_pred' not in df: raise RuntimeError(f'Design {did}: no y_pred')
        if not np.array_equal(fresh.astype(int), df['y_pred'].to_numpy(dtype=int)):
            raise RuntimeError(f'Design {did}: predictions do not match reconstructed feature order')
        if 'y_root' in df: truth = df['y_root'].to_numpy(dtype=int)
        elif 'y_true' in df: truth = df['y_true'].to_numpy(dtype=int)
        else:
            # Root saved predictions contain the class column named according to the runner.
            candidates = [c for c in df if c.startswith('y_root_') or c.startswith('class_')]
            if len(candidates) != 1: raise RuntimeError(f'Design {did}: cannot locate true class; columns={list(pred.columns)}')
            truth = df[candidates[0]].to_numpy(dtype=int)
        def score(z): return balanced_accuracy_score(truth, model.predict(z).astype(int))
        metric = 'balanced_accuracy'
    else:
        if 'delta_pred' not in df: raise RuntimeError(f'Design {did}: no delta_pred')
        if not np.allclose(fresh.astype(float), df['delta_pred'].to_numpy(dtype=float), rtol=1e-5, atol=1e-7):
            raise RuntimeError(f'Design {did}: predictions do not match reconstructed feature order')
        candidates = [c for c in df if c.startswith('delta_root') or c == 'delta_true']
        if len(candidates) != 1: raise RuntimeError(f'Design {did}: cannot locate true delta; columns={list(pred.columns)}')
        truth = df[candidates[0]].to_numpy(dtype=float)
        def score(z): return -mean_absolute_error(truth, model.predict(z).astype(float))
        metric = 'negative_MAE'
    base_score = float(score(X))
    rows=[]
    for feature in columns:
        declines=[]
        for _ in range(REPEATS):
            xp=X.copy()
            xp[feature]=rng.permutation(xp[feature].to_numpy())
            declines.append(base_score - float(score(xp)))
        rows.append({'design_id':did,'comparison':comp,'train_setting':setting,'task':task,
                     'model':model_name,'feature':feature,'importance_mean':float(np.mean(declines)),
                     'importance_std':float(np.std(declines,ddof=1)),
                     'importance_median':float(np.median(declines)),
                     'evaluation_metric':metric,'baseline':base_score,'n_test':len(X),
                     'repeats':REPEATS})
    table=pd.DataFrame(rows).sort_values('importance_mean',ascending=False)
    stem=f'root_design_{did:02d}_{comp}_{setting}_{task}'
    table.to_csv(OUT/f'{stem}_all_64_features.csv',index=False)
    top=table.head(TOP_K).iloc[::-1]
    fig,ax=plt.subplots(figsize=(10,7))
    ax.barh(top['feature'],top['importance_mean'],xerr=top['importance_std'],capsize=2)
    ax.axvline(0,color='gray',lw=1)
    ax.set_xlabel('Mean drop in '+metric+' when feature is permuted')
    ax.set_title(f'{comp} | {setting} | {task} | {model_name}')
    fig.tight_layout(); fig.savefig(OUT/f'{stem}_top15.png',dpi=200);plt.close(fig)
    return table,dict(design_id=did,comparison=comp,train_setting=setting,task=task,
                      selected_model=model_name,selection_metric=select_metric,
                      selection_score=select_value,importance_metric=metric,
                      baseline=base_score,n_test=len(X),top_feature=table.iloc[0]['feature'])

def main():
    root=get_root_run()
    manifest=pd.read_csv(root/'experiment_manifest.csv')
    results=pd.read_csv(root/'all_root_only_model_results.csv')
    needed={'design_id','comparison','train_setting','task'}
    if not needed.issubset(manifest.columns):
        raise RuntimeError(f'Unexpected experiment manifest columns: {list(manifest.columns)}')
    if len(manifest)!=8 or set(manifest['design_id'].astype(int))!=set(range(1,9)):
        raise RuntimeError('Expected final eight root-node designs; refusing older 4-design runs')
    if not FEATURES_FILE.is_file(): raise FileNotFoundError(FEATURES_FILE)
    static=add_key(pd.read_excel(FEATURES_FILE))
    OUT.mkdir(parents=True,exist_ok=False)
    all_data=[]; summaries=[]; rng=np.random.default_rng(SEED)
    for _,entry in manifest.sort_values('design_id').iterrows():
        print('Processing root design',entry['design_id'],entry['comparison'],entry['train_setting'],entry['task'],flush=True)
        table,summary=one_design(root,static,entry,results,rng)
        all_data.append(table);summaries.append(summary)
        pd.concat(all_data,ignore_index=True).to_csv(OUT/'all_root_feature_importance.csv',index=False)
        pd.DataFrame(summaries).to_csv(OUT/'root_feature_importance_summary.csv',index=False)
        print(' -> ',summary['selected_model'],':',table['feature'].head(5).tolist(),flush=True)
    print('COMPLETE:',OUT)

if __name__=='__main__':
    main()


Selected most recent completed root-run: C:\Users\suuser\Desktop\minlplib\Areview\combined_minlplib_qplib\sklearn_root_8_designs_278_two_excluded_20261022_201010_349221
Processing root design 1 LP_vs_SDP combined_to_combined classification
 ->  XGB-C : ['eta_A0_over_n', 'PH_A0', 'lambda_max_A0_scaled', 'h_avg_Lambda_max_scaled', 'h_max_Rho']
Processing root design 2 LP_vs_SDP combined_to_combined regression
 ->  RFR : ['eta_A0_over_n', 'theta_A0', 'PH_A0', 'rho_A0_over_n', 'lambda_min_A0_scaled']
Processing root design 3 LP_vs_SDP synthetic_to_combined classification
 ->  XGB-C : ['h_max_Lambda_min_scaled', 'BoundsExist', 'lambda_max_A0_scaled', 'h_IQR_LogCond', 'g_indef_Theta']
Processing root design 4 LP_vs_SDP synthetic_to_combined regression
 ->  RFR : ['theta_A0', 'lambda_min_A0_scaled', 'eta_A0_over_n', 'g_indef_Theta', 'h_max_Theta']
Processing root design 5 LP_vs_SDP_PRIME combined_to_combined classification
 ->  XGB-C : ['lambda_max_A0_scaled', 'lambda_min_A0_scaled', 'eta_A0_